<a href="https://colab.research.google.com/github/mbrennan5/LSTM-TREND/blob/claude%2Fplan-session-VX3Ru/LSTM_TREND_FAMILY_feature_lock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# SOVEREIGN TITAN v3.19.18 — GPU + SPEED EDITION v2
# Speed changes vs prior version:
#   1. Parallel yfinance downloads       (ThreadPoolExecutor)
#   2. Sequences built ONCE per iter     (was rebuilt 150× per feature)
#   3. Single model + perm-importance    (1 train + 150 forward passes vs 150 trains)
#   4. tf.data pipeline with prefetch    (GPU never idles waiting for CPU)
#   5. @tf.function on eval step         (JIT-compiles permutation scoring)
# ── NEW IN THIS VERSION ──────────────────────────────────────────────────────
#   6. ALL rolling functions JIT-compiled via Numba  (no more Python lambdas)
#      _lin_slope, _hurst, _cog, _shannon, _r_sq, _wma — all native machine code
#   7. ProcessPoolExecutor for feature generation    (true multi-core, bypasses GIL)
#   8. Duplicate linreg/slope computation removed
#   9. Dispersion vectorised (np.stack instead of Python list comprehension)
#  10. BRAIN_LOCKS names corrected to match actual column output
# ==============================================================================
# ### BLOCK 0: GPU SETUP
# ==============================================================================
import os, gc, warnings
warnings.filterwarnings('ignore')
import tensorflow as tf

def setup_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("⚠️  No GPU — running CPU. Colab: Runtime → Change runtime type → T4 GPU")
        return False
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f"✅ Mixed precision: {tf.keras.mixed_precision.global_policy().name}")
    with tf.device('/device:GPU:0'):
        _ = tf.random.normal((10, 10)) @ tf.random.normal((10, 10))
    print(f"✅ GPU confirmed: {tf.test.gpu_device_name()}")
    return True

GPU_AVAILABLE = setup_gpu()
DEVICE = '/device:GPU:0' if GPU_AVAILABLE else '/cpu:0'
print(f"[SYSTEM] Active compute device: {DEVICE}\n")


# ==============================================================================
# ### BLOCK 1: SYSTEM INITIALIZATION
# ==============================================================================
import numpy as np, pandas as pd, yfinance as yf
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import random
from numba import jit
from datetime import datetime
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

TEST_NAME        = "Sovereign_Titan_v3.19.18_Entropy_Injection"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR):
    os.makedirs(OUTPUT_DRIVE_DIR)

# Number of CPU workers for feature generation
# Colab free = 2, Colab Pro = 4-8. Auto-detects.
N_FEATURE_WORKERS = max(1, (os.cpu_count() or 2))
print(f"[SYSTEM] Feature generation workers: {N_FEATURE_WORKERS}")

TITAN_SYMBOLS = [
    'AA','AAL','AAPL','ABNB','ACWI','AEM','AFRM','AI','ALAB','ALB','AMAT','AMD','AMZN',
    'ANET','APA','APH','ARKK','AVGO','BA','BABA','BAC','BKR','BLDR','C','CARR','CAT',
    'CCJ','CCL','CE','CELH','CLF','CLSK','CMG','CNC','CPRT','CRM','CSCO','CSX','CVS',
    'CVX','DAL','DDOG','DHR','DIA','DIS','DKNG','DLTR','DOW','DVN','DXCM','EA','EBAY',
    'EEM','EMR','EQT','EWJ','EWT','EWW','EWY','EWZ','EXC','F','FANG','FCX','FITB',
    'FTNT','FTV','FXI','GBTC','GDX','GDXJ','GEHC','GFS','GIS','GOOG','GOOGL','GS',
    'HAL','HOOD','HPE','HPQ','HWM','IAU','IBM','IGV','IJH','IJR','INTC','IP','IR',
    'IWM','IYR','JNJ','KDP','KMI','KO','KRE','KWEB','LOW','LRCX','LUV','LVS','LYFT',
    'MAR','MARA','MCHP','MGM','MNST','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR',
    'MU','NCLH','NEE','NEM','NKE','NUE','NVDA','NVO','NXPI','ON','ORCL','OXY','PANW',
    'PCAR','PDD','PEP','PFE','PINS','PLTR','PYPL','QCOM','QQQ','QQQM','RBLX','RIOT',
    'RIVN','RTX','SBUX','SCHW','SHOP','SJM','SLB','SLV','SMCI','SMH','SNAP','SNOW',
    'SOFI','SOXX','SPLG','SPY','TER','TGT','TJX','TLT','TMUS','TQQQ','TSCO','TSLA',
    'TTD','TTWO','TWLO','TXN','U','UAL','UBER','UPS','USB','USO','VLO','VNQ','VRT',
    'VST','VT','VTR','WMT','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP',
    'XLRE','XLU','XLV','XLY','XOM','XOP','XRT'
]


# ==============================================================================
# ### BLOCK 2: NUMBA JIT ROLLING KERNELS
# cache=True saves compiled artifacts to disk — subsequent runs skip recompile.
# Each function replaces a pandas rolling().apply(lambda...) call.
# Speedup per function: ~10-50× over interpreted Python lambdas.
# ==============================================================================

# ── Linear slope (replaces np.polyfit inside rolling) ─────────────────────────
@jit(nopython=True, cache=True)
def _lin_slope_nb(y):
    """OLS slope — equivalent to np.polyfit(x, y, 1)[0] but ~20× faster."""
    n = len(y)
    if n < 2: return 0.0
    x_mean = (n - 1) / 2.0
    y_mean = 0.0
    for i in range(n): y_mean += y[i]
    y_mean /= n
    num = 0.0; den = 0.0
    for i in range(n):
        dx = i - x_mean
        num += dx * (y[i] - y_mean)
        den += dx * dx
    return num / den if den != 0.0 else 0.0

@jit(nopython=True, cache=True)
def _rolling_linslope(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _lin_slope_nb(arr[i - window + 1 : i + 1])
    return out


# ── Hurst exponent ─────────────────────────────────────────────────────────────
@jit(nopython=True, cache=True)
def _hurst_nb(y):
    n = len(y)
    if n < 2: return 0.5
    mean = 0.0
    for i in range(n): mean += y[i]
    mean /= n
    var = 0.0
    for i in range(n): var += (y[i] - mean) ** 2
    std = (var / n) ** 0.5
    if std < 1e-12: return 0.5
    r = np.log(std + 1e-9) / np.log(n)
    return r if not np.isnan(r) else 0.5

@jit(nopython=True, cache=True)
def _rolling_hurst(arr, window):
    n = len(arr); out = np.full(n, 0.5)
    for i in range(window - 1, n):
        out[i] = _hurst_nb(arr[i - window + 1 : i + 1])
    return out


# ── Center of Gravity ──────────────────────────────────────────────────────────
@jit(nopython=True, cache=True)
def _cog_nb(y):
    n = len(y)
    if n < 2: return 0.0
    num = 0.0; den = 0.0
    for i in range(n):
        w = float(i + 1)
        num += w * y[i]
        den += y[i]
    return -num / (den + 1e-9)

@jit(nopython=True, cache=True)
def _rolling_cog(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _cog_nb(arr[i - window + 1 : i + 1])
    return out


# ── Shannon entropy (manual histogram — np.histogram not in nopython) ──────────
@jit(nopython=True, cache=True)
def _shannon_nb(y, bins=10):
    n = len(y)
    if n < 2: return 0.0
    mn = y[0]; mx = y[0]
    for i in range(1, n):
        if y[i] < mn: mn = y[i]
        if y[i] > mx: mx = y[i]
    if mx == mn: return 0.0
    counts = np.zeros(bins)
    for i in range(n):
        idx = int((y[i] - mn) / (mx - mn) * bins)
        if idx >= bins: idx = bins - 1
        counts[idx] += 1.0
    entropy = 0.0
    for i in range(bins):
        p = counts[i] / n + 1e-9
        entropy -= p * np.log(p)
    return entropy

@jit(nopython=True, cache=True)
def _rolling_shannon(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _shannon_nb(arr[i - window + 1 : i + 1])
    return out


# ── R-squared (correlation² of index vs values) ────────────────────────────────
@jit(nopython=True, cache=True)
def _r_sq_nb(y):
    n = len(y)
    if n < 2: return 0.0
    x_mean = (n - 1) / 2.0
    y_mean = 0.0
    for i in range(n): y_mean += y[i]
    y_mean /= n
    num = 0.0; den_x = 0.0; den_y = 0.0
    for i in range(n):
        dx = i - x_mean; dy = y[i] - y_mean
        num   += dx * dy
        den_x += dx * dx
        den_y += dy * dy
    if den_x == 0.0 or den_y == 0.0: return 0.0
    r = num / ((den_x ** 0.5) * (den_y ** 0.5))
    return r * r

@jit(nopython=True, cache=True)
def _rolling_r_sq(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _r_sq_nb(arr[i - window + 1 : i + 1])
    return out


# ── Weighted Moving Average ────────────────────────────────────────────────────
@jit(nopython=True, cache=True)
def _rolling_wma(arr, window):
    n = len(arr); out = np.full(n, np.nan)
    w_sum = window * (window + 1) / 2.0
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += arr[i - window + 1 + j] * (j + 1)
        out[i] = s / w_sum
    return out


# ── Kalman filter (unchanged) ──────────────────────────────────────────────────
@jit(nopython=True, cache=True)
def _kalman_numba(price, r=0.0001, q=0.001):
    x_hat = np.zeros_like(price); p = np.zeros_like(price)
    x_hat[0] = price[0]; p[0] = 1.0
    for t in range(1, len(price)):
        p_minus  = p[t-1] + q
        k        = p_minus / (p_minus + r)
        x_hat[t] = x_hat[t-1] + k * (price[t] - x_hat[t-1])
        p[t]     = (1 - k) * p_minus
    return x_hat


# ── Trigger first-time Numba compilation at import time (not during the run) ───
def _warm_up_numba():
    dummy = np.random.randn(60).astype(np.float64)
    _rolling_linslope(dummy, 10)
    _rolling_hurst(dummy, 50)
    _rolling_cog(dummy, 20)
    _rolling_shannon(dummy, 20)
    _rolling_r_sq(dummy, 30)
    _rolling_wma(dummy, 10)
    _kalman_numba(dummy)
    print("✅ Numba kernels compiled and ready")

_warm_up_numba()


# ==============================================================================
# ### BLOCK 3: FEATURE FACTORY — now uses Numba kernels throughout
# Top-level function required for ProcessPoolExecutor pickling.
# ==============================================================================
def generate_factory_features_v2(df):
    df = df.copy()
    df['hlc3']    = (df['high'] + df['low'] + df['close']) / 3
    df['T_FINAL'] = (df['close'].shift(-1) > df['close']).astype('float')
    df.loc[df.index[-1], 'T_FINAL'] = np.nan  # last row has no future close

    hlc = df['hlc3'].values.astype(np.float64)
    hi  = df['high'].values.astype(np.float64)
    lo  = df['low'].values.astype(np.float64)
    cl  = df['close'].values.astype(np.float64)
    vol = df['volume'].values.astype(np.float64)
    idx = df.index

    # ── Moving averages ────────────────────────────────────────────────────────
    ema30 = pd.Series(hlc, index=idx).ewm(span=30).mean().values
    ema30_2 = pd.Series(ema30, index=idx).ewm(span=30).mean().values
    ema30_3 = pd.Series(ema30_2, index=idx).ewm(span=30).mean().values
    tema_30 = 3*ema30 - 3*ema30_2 + ema30_3

    sma_20 = pd.Series(hlc, index=idx).rolling(20).mean().values

    # WMA via Numba — replaces two rolling().apply(lambda) calls
    wma1   = _rolling_wma(hlc, 10)
    wma2   = _rolling_wma(hlc, 21)
    hma_raw = 2 * wma1 - wma2
    hma_21  = pd.Series(hma_raw, index=idx).rolling(5).mean().values

    kalman = _kalman_numba(hlc)

    # ── Efficiency / trend strength ────────────────────────────────────────────
    hlc_s = pd.Series(hlc, index=idx)
    er_20        = (hlc_s.diff(20).abs() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values
    vidya_cmo_20 = (hlc_s.diff().rolling(20).sum() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values

    # ── Numba rolling functions ────────────────────────────────────────────────
    r_sq_30    = _rolling_r_sq(hlc, 30)       # was: rolling().apply(corrcoef lambda)
    hurst_50   = _rolling_hurst(hlc, 50)      # was: rolling().apply(_hurst lambda)
    shannon_20 = _rolling_shannon(hlc, 20)    # was: rolling().apply(histogram lambda)
    cog_20     = _rolling_cog(hlc, 20)        # was: rolling().apply(_cog lambda)

    # linreg and logistic — computed ONCE (was computed twice — duplicate removed)
    linreg_30       = _rolling_linslope(hlc, 30)
    slope_std       = np.nanstd(linreg_30) + 1e-9
    logistic_prob_30 = 1.0 / (1.0 + np.exp(-linreg_30 / slope_std))

    # ── MTSI (Money Trend Strength Indicator) ─────────────────────────────────
    # Formula: EMA(3) of (close - 2-bar VWAP)
    # 2-bar VWAP = sum(hlc3 * volume, 2) / sum(volume, 2)
    # Measures how far close sits above/below the short-term volume-weighted
    # price. Oscillates around zero in price units — z-lens normalises
    # cross-stock price scaling, same rationale as lr_slope_30.
    _tp_v  = pd.Series(hlc * vol, index=idx)
    _vol_s = pd.Series(vol, index=idx)
    _cl_s  = pd.Series(cl,  index=idx)
    mtsi   = (_cl_s - (_tp_v.rolling(2).sum() / (_vol_s.rolling(2).sum() + 1e-9)))              .ewm(span=3).mean().values

    # ── ADX ────────────────────────────────────────────────────────────────────
    cl_prev = np.roll(cl, 1); cl_prev[0] = cl[0]
    tr      = np.maximum(hi-lo, np.maximum(np.abs(hi-cl_prev), np.abs(lo-cl_prev)))
    atr_14  = pd.Series(tr, index=idx).rolling(14).mean().values
    hi_prev = np.roll(hi, 1); hi_prev[0] = hi[0]
    lo_prev = np.roll(lo, 1); lo_prev[0] = lo[0]
    plus_dm  = np.where((hi-hi_prev) > (lo_prev-lo), np.maximum(hi-hi_prev, 0), 0).astype(np.float64)
    minus_dm = np.where((lo_prev-lo) > (hi-hi_prev), np.maximum(lo_prev-lo, 0), 0).astype(np.float64)
    pdi14 = 100 * (pd.Series(plus_dm,index=idx).rolling(14).mean() / (pd.Series(atr_14,index=idx)+1e-9))
    mdi14 = 100 * (pd.Series(minus_dm,index=idx).rolling(14).mean() / (pd.Series(atr_14,index=idx)+1e-9))
    adx_14 = (100 * np.abs(pdi14-mdi14) / (pdi14+mdi14+1e-9)).rolling(14).mean().values

    # ── Dispersion — vectorised ────────────────────────────────────────────────
    sma_30 = pd.Series(hlc, index=idx).rolling(30).mean().values
    d_sma  = hlc / (sma_30  + 1e-9) - 1
    d_tema = hlc / (tema_30 + 1e-9) - 1
    d_kal  = hlc / (kalman  + 1e-9) - 1
    dispersion_30 = np.std(np.stack([d_sma, d_tema, d_kal], axis=1), axis=1)

    # ── Donchian / Aroon (VPIN removed) ───────────────────────────────────────
    hi_s = pd.Series(hi, index=idx)
    donchian_high_50 = (hi_s / hi_s.rolling(50).max() - 1).values
    aroon_up_25      = hi_s.rolling(25).apply(lambda x: float(np.argmax(x))/25, raw=True).values

    # ══════════════════════════════════════════════════════════════════════════
    # INDICATOR CLASSIFICATION
    #
    # RULE: if the signal is bounded or can be transformed into a bounded,
    #       mean-reverting series → z-lens (LENS 10 & 90).
    #       if the signal is genuinely unbounded/cumulative in price units
    #       and no natural normalisation exists → rolling % (WIN 10, 30, 90).
    #
    # Z-LENS GROUP (16 indicators × 2 lenses × 3 transforms = 96 features)
    # ─────────────────────────────────────────────────────────────────────
    #   Raw bounded indicators (passed directly):
    #     er_20, vidya_cmo_20, r_sq_30, hurst_50, shannon_20, adx_14,
    #     logistic_prob_30, aroon_up_25, donchian_high_50, dispersion_30,
    #     lr_slope_30
    #
    #   Price MA indicators (pre-transformed to hlc3/MA - 1 first):
    #     tema_30, sma_20, hma_21, kalman
    #     hlc3/MA - 1 is mean-reverting around zero → bounded → z-lens natural.
    #     Transform happens before the lens, not instead of it.
    #
    # ROLLING % GROUP (1 indicator × 3 windows = 3 features)
    # ─────────────────────────────────────────────────────────────────────
    #   cog_20: Center of Gravity is in price units, unbounded.
    #           cog / rolling_mean(cog, N) - 1 measures deviation from norm.
    #
    # TOTAL: 90 + 3 = 93 features
    # ══════════════════════════════════════════════════════════════════════════

    # ── Pre-transform Price MAs: hlc3 / MA - 1 ────────────────────────────────
    # Result is a bounded, mean-reverting % deviation — ready for z-lens.
    tema_30_pct = hlc / (tema_30 + 1e-9) - 1
    sma_20_pct  = hlc / (sma_20  + 1e-9) - 1
    hma_21_pct  = hlc / (hma_21  + 1e-9) - 1
    kalman_pct  = hlc / (kalman  + 1e-9) - 1

    # ── Z-lens group ──────────────────────────────────────────────────────────
    Z_LENS_INDICATORS = {
        # Bounded oscillators — raw value
        'er_20':            pd.Series(er_20,            index=idx),
        'vidya_cmo_20':     pd.Series(vidya_cmo_20,     index=idx),
        'r_sq_30':          pd.Series(r_sq_30,          index=idx),
        'hurst_50':         pd.Series(hurst_50,         index=idx),
        'shannon_20':       pd.Series(shannon_20,       index=idx),
        'adx_14':           pd.Series(adx_14,           index=idx),
        'logistic_prob_30': pd.Series(logistic_prob_30, index=idx),
        'aroon_up_25':      pd.Series(aroon_up_25,      index=idx),
        'donchian_high_50': pd.Series(donchian_high_50, index=idx),
        'dispersion_30':    pd.Series(dispersion_30,    index=idx),
        'lr_slope_30':      pd.Series(linreg_30,        index=idx),
        # Price MAs — pre-transformed to hlc3/MA - 1
        'tema_30_pct':      pd.Series(tema_30_pct,      index=idx),
        'sma_20_pct':       pd.Series(sma_20_pct,       index=idx),
        'hma_21_pct':       pd.Series(hma_21_pct,       index=idx),
        'kalman_pct':       pd.Series(kalman_pct,       index=idx),
        # Price-unit oscillators — z-lens normalises cross-stock scaling
        'mtsi':             pd.Series(mtsi,             index=idx),
    }

    # ── Apply LENS 10 & 90: z, z_slope, z_sos on top of each indicator ────────
    for name, ind in Z_LENS_INDICATORS.items():
        arr = ind.values.astype(np.float64)
        for lens in [10, 90]:
            rm   = pd.Series(arr, index=idx).rolling(lens).mean().values
            rs   = pd.Series(arr, index=idx).rolling(lens).std().values
            z    = (arr - rm) / (rs + 1e-9)
            zs   = _rolling_linslope(z, lens)
            zsos = _rolling_linslope(zs, lens)
            df[f'LENS_{lens}_{name}_z']       = z
            df[f'LENS_{lens}_{name}_z_slope'] = zs
            df[f'LENS_{lens}_{name}_z_sos']   = zsos

    # ── Rolling % group: COG only ──────────────────────────────────────────────
    # COG is in price units — unbounded. Deviation from its own rolling mean
    # is the most natural normalisation available.
    cog_arr = cog_20.astype(np.float64)
    for win in [10, 30, 90]:
        rm = pd.Series(cog_arr, index=idx).rolling(win).mean().values
        df[f'WIN_{win}_cog_20_pct'] = cog_arr / (rm + 1e-9) - 1

    return df.replace([np.inf,-np.inf], np.nan).ffill().dropna(subset=['T_FINAL']).fillna(0)






# ── Top-level worker for ProcessPoolExecutor (must be picklable) ───────────────
def _process_symbol_worker(args):
    """Called in a subprocess. Returns processed DataFrame or None."""
    symbol, raw_dict = args
    try:
        raw_df = pd.DataFrame(raw_dict)
        raw_df.index = pd.to_datetime(raw_df.index)
        if len(raw_df) < 120:
            return None
        processed = generate_factory_features_v2(raw_df)
        if processed.empty:
            return None
        processed['symbol'] = symbol
        return processed.reset_index()          # reset so index survives pickling
    except Exception:
        return None


# ==============================================================================
# ### BLOCK 4: PARALLEL LOADER
# Phase 1: parallel network download (ThreadPoolExecutor)
# Phase 2: parallel feature generation (ProcessPoolExecutor — true multi-core)
# ==============================================================================
def fetch_data(symbol):
    try:
        data = yf.download(symbol, period="2y", interval="1d", progress=False)
        if data.empty: return None
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data.columns = [str(c).lower() for c in data.columns]
        return data if 'close' in data.columns else None
    except Exception:
        return None


def load_hybrid_data_parallel(brain_name, symbol_list, dl_workers=20):
    print(f"📥 Parallel download: {len(symbol_list)} symbols...")
    raw_results = {}

    # ── Phase 1: parallel I/O ──────────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=dl_workers) as pool:
        fut_map = {pool.submit(fetch_data, sym): sym for sym in symbol_list}
        for fut in tqdm(as_completed(fut_map), total=len(symbol_list), desc="⬇ Downloading"):
            sym  = fut_map[fut]
            data = fut.result()
            if data is not None and len(data) >= 120:
                raw_results[sym] = data

    print(f"   ✅ {len(raw_results)}/{len(symbol_list)} symbols fetched")
    if not raw_results:
        return pd.DataFrame()

    # ── Phase 2: parallel feature generation ──────────────────────────────────
    # DataFrames aren't directly picklable with their DatetimeIndex in all envs,
    # so we pass them as dicts and reconstruct inside the worker.
    work_items = [(sym, df.to_dict()) for sym, df in raw_results.items()]

    all_data = []
    print(f"⚙ Building features in parallel (workers={N_FEATURE_WORKERS})...")

    try:
        with ProcessPoolExecutor(max_workers=N_FEATURE_WORKERS) as pool:
            futures = {pool.submit(_process_symbol_worker, item): item[0]
                       for item in work_items}
            for fut in tqdm(as_completed(futures), total=len(work_items),
                            desc="⚙ Features"):
                result = fut.result()
                if result is not None:
                    result = result.set_index(result.columns[0])   # restore index
                    all_data.append(result)

    except Exception as e:
        # Fallback: sequential (safe in any Colab environment)
        print(f"  ⚠️  ProcessPool failed ({e}) — falling back to sequential")
        for item in tqdm(work_items, desc="⚙ Features (sequential)"):
            result = _process_symbol_worker(item)
            if result is not None:
                result = result.set_index(result.columns[0])
                all_data.append(result)

    if not all_data:
        print("❌ No valid data after feature generation.")
        return pd.DataFrame()

    print(f"   ✅ {len(all_data)} symbols processed")
    return pd.concat(all_data, axis=0)



# ==============================================================================
# ### BLOCK 5: GPU-ACCELERATED AUDIT — single model + permutation importance
# ==============================================================================
def build_full_model(model_type, n_features, seq_len, device=DEVICE):
    with tf.device(device):
        model = Sequential([
            Input(shape=(seq_len, n_features)),
            GRU(128, return_sequences=True)  if model_type == 'GRU' else
            LSTM(128, return_sequences=True),
            Dropout(0.2),
            GRU(64)  if model_type == 'GRU' else LSTM(64),
            Dropout(0.2),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid', dtype='float32')
        ])
        model.compile(optimizer=Adam(1e-3),
                      loss='binary_crossentropy', metrics=['accuracy'])
    return model


@tf.function
def _eval_accuracy(model, X_batch, y_batch):
    preds   = tf.squeeze(model(X_batch, training=False), axis=-1)
    correct = tf.equal(tf.cast(preds >= 0.5, tf.int32), tf.cast(y_batch, tf.int32))
    return tf.reduce_mean(tf.cast(correct, tf.float32))


def run_judicial_audit(brain_name, master_df, model_type='GRU',
                       seq_len=10, epochs=5, batch_size=1024):
    feature_cols = [c for c in master_df.columns if c.startswith('LENS_') or c.startswith('WIN_')]
    n_features   = len(feature_cols)

    y_raw = master_df['T_FINAL'].values.astype(np.float32)
    X_raw = master_df[feature_cols].values

    n     = len(X_raw)
    n_seq = n - seq_len
    split = int(n_seq * 0.8)

    # Fit scaler only on rows used to build training sequences (no leakage)
    scaler = RobustScaler()
    scaler.fit(X_raw[:seq_len + split])
    X_scaled = scaler.transform(X_raw).astype(np.float32)

    X_seqs = np.stack([X_scaled[i-seq_len:i] for i in range(seq_len, n)])
    y_seqs = y_raw[seq_len:]

    X_tr, X_val  = X_seqs[:split], X_seqs[split:]
    y_tr, y_val  = y_seqs[:split], y_seqs[split:]

    print(f"  [DATA] train={len(X_tr):,}  val={len(X_val):,}  features={n_features}")

    AUTO = tf.data.AUTOTUNE
    train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
                .shuffle(min(20_000, len(X_tr)), reshuffle_each_iteration=True)
                .batch(batch_size).prefetch(AUTO))
    val_ds   = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
                .batch(batch_size * 2).prefetch(AUTO))

    model = build_full_model(model_type, n_features, seq_len)

    with tf.device(DEVICE):
        model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=6,
                                           restore_best_weights=True),
                             ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=3, min_lr=1e-5)],
                  verbose=1)

    X_val_tf     = tf.constant(X_val)
    y_val_tf     = tf.constant(y_val)
    baseline_acc = _eval_accuracy(model, X_val_tf, y_val_tf).numpy()
    print(f"  [MODEL] Baseline val accuracy: {baseline_acc:.4f}")

    report_rows = []
    for fi, feat_name in enumerate(tqdm(feature_cols, desc="Permutation scoring")):
        try:
            X_perm = X_val.copy()
            flat   = X_perm[:, :, fi].flatten()
            np.random.shuffle(flat)
            X_perm[:, :, fi] = flat.reshape(X_perm[:, :, fi].shape)
            perm_acc = _eval_accuracy(model, tf.constant(X_perm), y_val_tf).numpy()
            report_rows.append({'Feature': feat_name,
                                 'I_raw': max(0.0, baseline_acc - perm_acc)})
        except Exception:
            report_rows.append({'Feature': feat_name, 'I_raw': 0.0})

    del model; gc.collect(); tf.keras.backend.clear_session()
    return pd.DataFrame(report_rows)


# ==============================================================================
# ### BLOCK 6: SOVEREIGN HUNT & DIVERSITY ANCHORS
# BRAIN_LOCKS corrected to match actual factory column names
# ==============================================================================
BRAIN_LOCKS = {
    'DIRECTION': ['LENS_90_cog_20_z_slope'],
    'EASE':      ['LENS_90_cog_20_z_sos'],
    'EXP':       ['LENS_10_hurst_50_z', 'LENS_90_cog_20_z_sos']
}
# Bounded indicators:   LENS_{10|90}_{name}_{z|z_slope|z_sos}
# Unbounded indicators: WIN_{10|30|90}_{name}_pct


def _parse_feature_name(f):
    """
    Splits a LENS_ or WIN_ feature name into components.

    LENS_10_cog_20_z_slope  -> prefix='LENS', window='10', lookback='LENS_10_cog_20', family='cog'
    LENS_90_cog_20_z        -> prefix='LENS', window='90', lookback='LENS_90_cog_20', family='cog'
    WIN_10_cog_20_pct       -> prefix='WIN',  window='10', lookback='WIN_10_cog_20',  family='cog'

    RULE (same for LENS and WIN):
      - Only ONE window per family is allowed in the final 19.
        LENS_10 vs LENS_90 of cog_20 are competing — hunt picks the higher-impact one.
      - All three transforms of the winning window CAN coexist:
        LENS_10_cog_20_z, LENS_10_cog_20_z_slope, LENS_10_cog_20_z_sos share
        the same lookback key ('LENS_10_cog_20') so they don't block each other.
    """
    TRANSFORM_TOKENS = {'z', 'slope', 'sos', 'pct'}

    if f.startswith('LENS_') or f.startswith('WIN_'):
        parts     = f.split('_')
        prefix    = parts[0]
        window    = parts[1]
        remainder = list(parts[2:])

        while remainder and remainder[-1] in TRANSFORM_TOKENS:
            remainder.pop()

        indicator = '_'.join(remainder)
        family    = '_'.join(p for p in remainder if not p.isdigit())

        # For BOTH LENS and WIN: window is a competing choice.
        # Fold prefix+window into lookback so the family rule enforces
        # "only one window per indicator" in the hunt.
        # The three transforms (z, z_slope, z_sos) of the winning window
        # share the same lookback key and can all enter freely.
        lookback = f'{prefix}_{window}_{indicator}'

        return prefix, window, lookback, family

    return None, None, f, f


def apply_sovereign_hunt(ledger_df, master_data_df, brain_name, max_slots=19):
    from collections import Counter
    locked_list  = BRAIN_LOCKS.get(brain_name, [])
    candidates   = ledger_df.sort_values(by='I_raw', ascending=False)
    picked       = [f for f in locked_list if f in ledger_df['Feature'].values]

    # Warn loudly if a locked feature is missing — easier to catch than silent skip
    for lf in locked_list:
        if lf not in ledger_df['Feature'].values:
            print(f"  ⚠️  BRAIN_LOCK '{lf}' not found in feature columns — check name")

    CORR_THRESHOLD = 0.85

    # Track which LOOKBACK is in use per FAMILY (not a count — a specific value)
    # e.g. family_lookback['cog'] = 'cog_20'  → blocks 'cog_30' but not more 'cog_20' lenses
    family_lookback = {}
    for f in picked:
        _, _, lookback, family = _parse_feature_name(f)
        if family not in family_lookback:
            family_lookback[family] = lookback

    # Corr matrix covers both LENS_ and WIN_ columns
    feat_cols   = [c for c in master_data_df.columns if c.startswith('LENS_') or c.startswith('WIN_')]
    corr_matrix = master_data_df[feat_cols].corr()

    for _, row in candidates.iterrows():
        if len(picked) >= max_slots: break
        f_name    = row['Feature']
        f_impact  = row['I_Norm']
        if f_name in picked: continue

        _, _, f_lookback, f_family = _parse_feature_name(f_name)

        # Block if this family already has a DIFFERENT lookback committed
        if f_family in family_lookback and family_lookback[f_family] != f_lookback:
            continue

        # Correlation guard
        if len(picked) > 0 and corr_matrix[f_name].loc[picked].max() > CORR_THRESHOLD:
            continue

        picked.append(f_name)
        if f_family not in family_lookback:
            family_lookback[f_family] = f_lookback

    pca = PCA()
    pca.fit(RobustScaler().fit_transform(master_data_df[picked]))
    return picked, np.cumsum(pca.explained_variance_ratio_)


def generate_judicial_ledger(brain_name, report_df, master_data_df, iteration=1):
    df = report_df.copy()
    df['I_Norm']          = (df['I_raw'] - df['I_raw'].min()) / \
                            (df['I_raw'].max() - df['I_raw'].min() + 1e-9)
    active_picks, var_map = apply_sovereign_hunt(df, master_data_df, brain_name)
    corr_sub = master_data_df[active_picks].corr().abs()
    avg_corr = (corr_sub.sum().sum() - len(active_picks)) / \
               (len(active_picks)**2 - len(active_picks) + 1e-9)

    print(f"\n╔══ {brain_name} SOVEREIGN CORE V.3.19.18 (Iter {iteration}) ══╗")
    print(f"║ {'RNK':<3} | {'TREND FEATURE':<35} | {'UV%':<4} | {'mR':<4} | {'IMPACT':<8} ║")
    print("╠" + "═"*4 + "╬" + "═"*37 + "╬" + "═"*6 + "╬" + "═"*6 + "╬" + "═"*10 + "╣")

    for i, f_name in enumerate(active_picks):
        f_row       = df[df['Feature'] == f_name].iloc[0]
        is_locked   = f_name in BRAIN_LOCKS.get(brain_name, [])
        icon        = "🔒" if is_locked else "🔭"
        lb_val      = f_name.split('_')[1] if (f_name.startswith('LENS_') or f_name.startswith('WIN_')) else "??"
        other_picks = [p for p in active_picks if p != f_name]
        max_r  = corr_sub[f_name].loc[other_picks].max() if other_picks else 0.0
        uv_val = (1 - corr_sub[f_name].loc[other_picks].mean()) * 100 if other_picks else 100.0
        print(f"║ {i+1:02d}  | {icon} {f_name[:33]:<33} | {uv_val:>3.0f}% | {max_r:.2f} | {f_row['I_Norm']:.4f} ║")
        df.loc[df['Feature'] == f_name, ['UV%','Max_R','LB','Is_Locked']] = \
            [uv_val, max_r, lb_val, is_locked]

    total_var = var_map[-1] if len(var_map) > 0 else 0
    print("╠" + "═"*73 + "╣")
    print(f"║ PCA TOTAL VARIANCE RETENTION: {total_var*100:>33.2f}% ║")
    print(f"║ AVG TEAM CROSS-CORRELATION: {avg_corr:>35.3f} ║")
    print(f"║ SLOTS FILLED: {len(active_picks):>44}/19 ║")
    print("╚" + "═"*73 + "╝")
    return df[df['Feature'].isin(active_picks)]


# ==============================================================================
# ### BLOCK 7: COMMAND CENTER
# ==============================================================================
print("\n--- SOVEREIGN TITAN v3.19.18 — GPU + SPEED EDITION v2 ---")
choice        = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION','EASE','EXP'] if choice == '4' else \
                [{'1':'DIRECTION','2':'EASE','3':'EXP'}[choice]]
num_symbols   = int(input("Symbols per iteration (Default 50): ") or "50")
num_iters     = int(input("Iterations to run (Default 25): ")     or "25")

final_report_accumulator = []

for BRAIN in BRAINS_TO_RUN:
    CURRENT_MODEL_TYPE = 'GRU' if BRAIN == 'DIRECTION' else 'LSTM'
    print(f"\n[SYSTEM] Brain: {BRAIN} | Model: {CURRENT_MODEL_TYPE} | Device: {DEVICE}")

    for it in range(1, num_iters + 1):
        print(f"\n{'─'*55}")
        print(f"  Iteration {it}/{num_iters}  —  Brain: {BRAIN}")
        print(f"{'─'*55}")

        POOL      = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
        master_df = load_hybrid_data_parallel(BRAIN, POOL)
        if master_df.empty:
            print("  ⚠️  Empty master_df — skipping.")
            continue

        report_raw       = run_judicial_audit(BRAIN, master_df, model_type=CURRENT_MODEL_TYPE)
        iteration_ledger = generate_judicial_ledger(BRAIN, report_raw, master_df, iteration=it)

        iteration_ledger['Iteration']  = it
        iteration_ledger['Brain']      = BRAIN
        iteration_ledger['Model_Type'] = CURRENT_MODEL_TYPE
        iteration_ledger['Timestamp']  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        final_report_accumulator.append(iteration_ledger)

        gc.collect()
        tf.keras.backend.clear_session()


# ==============================================================================
# ### BLOCK 8: FINAL EXPORT & SOVEREIGN SELECTION
# ==============================================================================
if final_report_accumulator:
    raw_df = pd.concat(final_report_accumulator, axis=0)

    stats = (raw_df.groupby(['Brain','Feature'])
             .agg(Persistence=('Feature','count'),
                  A_Impact=('I_Norm','mean'),
                  A_UV=('UV%','mean'))
             .reset_index())

    final_df = (raw_df.merge(stats, on=['Brain','Feature'], how='left')
                      .sort_values(['Brain','Persistence','A_Impact'], ascending=False))

    report_filename = f"Sovereign_Audit_Master_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    report_path     = os.path.join(OUTPUT_DRIVE_DIR, report_filename)
    final_df.to_csv(report_path, index=False)

    print("\n" + "="*65)
    print("✅ GLOBAL AUDIT COMPLETE")
    print(f"📊 DATA ROWS COLLECTED: {len(raw_df)}")
    print(f"📂 CSV SAVED TO:        {report_path}")
    print("="*65)

    print("\n" + "═"*65)
    print("🚀 FINAL SOVEREIGN ARRAYS (TOP 19 PER BRAIN)")
    print("═"*65)
    FINAL_SELECTIONS = {}

    for brain in BRAINS_TO_RUN:
        brain_stats = (stats[stats['Brain'] == brain]
                       .sort_values(['Persistence','A_Impact'], ascending=False))
        top_19      = brain_stats.head(19)
        FINAL_SELECTIONS[brain] = top_19['Feature'].tolist()

        print(f"\n💎 FINAL 19 — BRAIN: {brain}")
        print(f"{'RNK':<3} | {'FEATURE':<38} | {'PERSIST':<8} | {'AVG_IMP':<8}")
        print("─" * 62)
        for i, row in top_19.reset_index(drop=True).iterrows():
            print(f"{i+1:02d}  | {row['Feature']:<38} | "
                  f"{int(row['Persistence']):>2}/{num_iters:<5} | {row['A_Impact']:.4f}")

    for brain, winners in FINAL_SELECTIONS.items():
        BRAIN_LOCKS[brain] = winners

    print("\n" + "═"*65)
    print("✅ FINAL 19 SYNCED TO BRAIN_LOCKS")
    print(f"📂 TOTAL UNIQUE FEATURES LOGGED: {len(stats)}")
    print("═"*65)

else:
    print("\n⚠️ [CRITICAL] No data collected. Audit failed.")

⚠️  No GPU — running CPU. Colab: Runtime → Change runtime type → T4 GPU
[SYSTEM] Active compute device: /cpu:0

[SYSTEM] Feature generation workers: 2
✅ Numba kernels compiled and ready

--- SOVEREIGN TITAN v3.19.18 — GPU + SPEED EDITION v2 ---
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 1
Symbols per iteration (Default 50): 15
Iterations to run (Default 25): 2

[SYSTEM] Brain: DIRECTION | Model: GRU | Device: /cpu:0

───────────────────────────────────────────────────────
  Iteration 1/2  —  Brain: DIRECTION
───────────────────────────────────────────────────────
📥 Parallel download: 15 symbols...


⬇ Downloading:   0%|          | 0/15 [00:00<?, ?it/s]

   ✅ 15/15 symbols fetched
⚙ Building features in parallel (workers=2)...


⚙ Features:   0%|          | 0/15 [00:00<?, ?it/s]

   ✅ 15 symbols processed
  [DATA] train=6,004  val=1,501  features=99
Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 412ms/step - accuracy: 0.5559 - loss: 0.6908 - val_accuracy: 0.6362 - val_loss: 0.6491 - learning_rate: 0.0010
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 324ms/step - accuracy: 0.6062 - loss: 0.6570 - val_accuracy: 0.6382 - val_loss: 0.6359 - learning_rate: 0.0010
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 364ms/step - accuracy: 0.6238 - loss: 0.6442 - val_accuracy: 0.6362 - val_loss: 0.6252 - learning_rate: 0.0010
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 676ms/step - accuracy: 0.6495 - loss: 0.6294 - val_accuracy: 0.6602 - val_loss: 0.6146 - learning_rate: 0.0010
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 317ms/step - accuracy: 0.6516 - loss: 0.6215 - val_accuracy: 0.6642 - val_loss: 0.6041 - learning_rate: 0.0010
  [MODEL] Baseline val accuracy: 0.6642


...

   ✅ 15/15 symbols fetched
⚙ Building features in parallel (workers=2)...


⚙ Features:   0%|          | 0/15 [00:00<?, ?it/s]

   ✅ 15 symbols processed
  [DATA] train=6,004  val=1,501  features=99
Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 517ms/step - accuracy: 0.5128 - loss: 0.7129 - val_accuracy: 0.6269 - val_loss: 0.6545 - learning_rate: 0.0010
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 544ms/step - accuracy: 0.6139 - loss: 0.6568 - val_accuracy: 0.6469 - val_loss: 0.6292 - learning_rate: 0.0010
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 340ms/step - accuracy: 0.6303 - loss: 0.6385 - val_accuracy: 0.6642 - val_loss: 0.6075 - learning_rate: 0.0010
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 321ms/step - accuracy: 0.6499 - loss: 0.6171 - val_accuracy: 0.6629 - val_loss: 0.5968 - learning_rate: 0.0010
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 299ms/step - accuracy: 0.6653 - loss: 0.6045 - val_accuracy: 0.6909 - val_loss: 0.5818 - learning_rate: 0.0010
  [MODEL] Baseline val accuracy: 0.6909


Permutation scoring:   0%|          | 0/99 [00:00<?, ?it/s]

  ⚠️  BRAIN_LOCK 'LENS_90_cog_20_z_slope' not found in feature columns — check name

╔══ DIRECTION SOVEREIGN CORE V.3.19.18 (Iter 2) ══╗
║ RNK | TREND FEATURE                       | UV%  | mR   | IMPACT   ║
╠════╬═════════════════════════════════════╬══════╬══════╬══════════╣
║ 01  | 🔭 LENS_90_vidya_cmo_20_z_sos        |  71% | 0.82 | 1.0000 ║
║ 02  | 🔭 LENS_90_hurst_50_z_sos            |  78% | 0.75 | 0.9490 ║
║ 03  | 🔭 LENS_90_r_sq_30_z_sos             |  78% | 0.75 | 0.4184 ║
║ 04  | 🔭 WIN_30_cog_20_pct                 |  98% | 0.05 | 0.2959 ║
║ 05  | 🔭 LENS_90_aroon_up_25_z_sos         |  77% | 0.81 | 0.1633 ║
║ 06  | 🔭 LENS_90_dispersion_30_z_slope     |  74% | 0.75 | 0.1429 ║
║ 07  | 🔭 LENS_90_vidya_cmo_20_z            |  76% | 0.77 | 0.1327 ║
║ 08  | 🔭 LENS_90_kalman_pct_z              |  91% | 0.59 | 0.1327 ║
║ 09  | 🔭 LENS_90_mtsi_z_sos                |  69% | 0.81 | 0.1224 ║
║ 10  | 🔭 LENS_90_sma_20_pct_z_slope        |  69% | 0.71 | 0.1122 ║
║ 11  | 🔭 LENS_90_tema_30_pct_z 

In [ ]:
# SOVEREIGN TITAN v3.20.2 — MFI Removed, PSAR on LinReg, Core Indicators Verified
# Universal: Works in Google Colab (with Drive) AND local machines
import os, gc, warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# ### ENVIRONMENT DETECTION & SETUP
# ==============================================================================

def detect_environment():
    """Detect if running in Colab or local"""
    try:
        import google.colab
        return 'COLAB'
    except ImportError:
        return 'LOCAL'

ENV = detect_environment()
print(f"[ENVIRONMENT] Running in: {ENV}")

# Mount Google Drive if in Colab
if ENV == 'COLAB':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_DIR = '/content/drive/MyDrive/Sovereign_Titan_Results/'
    print(f"✅ Google Drive mounted")
else:
    BASE_OUTPUT_DIR = './results/'

print(f"[STORAGE] Base output directory: {BASE_OUTPUT_DIR}\n")

# ==============================================================================
# ### GPU SETUP
# ==============================================================================

import tensorflow as tf

def setup_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("⚠️  No GPU detected — running on CPU (slower but functional)")
        if ENV == 'COLAB':
            print("    💡 Enable GPU: Runtime → Change runtime type → GPU")
        else:
            print("    💡 For GPU support: install CUDA 11.8 + compatible NVIDIA GPU")
        return False

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f"✅ Mixed precision: {tf.keras.mixed_precision.global_policy().name}")

    with tf.device('/device:GPU:0'):
        _ = tf.random.normal((10, 10)) @ tf.random.normal((10, 10))

    print(f"✅ GPU confirmed: {tf.test.gpu_device_name()}")
    return True

GPU_AVAILABLE = setup_gpu()
DEVICE = '/device:GPU:0' if GPU_AVAILABLE else '/cpu:0'
print(f"[SYSTEM] Active compute device: {DEVICE}\n")

# ==============================================================================
# ### SYSTEM INITIALIZATION
# ==============================================================================

import numpy as np, pandas as pd, yfinance as yf
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import random
from numba import jit
from datetime import datetime

# Test-specific output directory
TEST_NAME = "Sovereign_Titan_v3.20.2_CoreVerified"
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, TEST_NAME)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📁 Created output directory: {OUTPUT_DIR}")

N_FEATURE_WORKERS = max(1, (os.cpu_count() or 2) - 1)
print(f"[SYSTEM] Feature generation workers: {N_FEATURE_WORKERS}")
print(f"[SYSTEM] Total CPU cores: {os.cpu_count()}")
print(f"[SYSTEM] Results will be saved to: {OUTPUT_DIR}\n")

TITAN_SYMBOLS = [
    'AA','AAL','AAPL','ABNB','ACWI','AEM','AFRM','AI','ALAB','ALB','AMAT','AMD','AMZN',
    'ANET','APA','APH','ARKK','AVGO','BA','BABA','BAC','BKR','BLDR','C','CARR','CAT',
    'CCJ','CCL','CE','CELH','CLF','CLSK','CMG','CNC','CPRT','CRM','CSCO','CSX','CVS',
    'CVX','DAL','DDOG','DHR','DIA','DIS','DKNG','DLTR','DOW','DVN','DXCM','EA','EBAY',
    'EEM','EMR','EQT','EWJ','EWT','EWW','EWY','EWZ','EXC','F','FANG','FCX','FITB',
    'FTNT','FTV','FXI','GBTC','GDX','GDXJ','GEHC','GFS','GIS','GOOG','GOOGL','GS',
    'HAL','HOOD','HPE','HPQ','HWM','IAU','IBM','IGV','IJH','IJR','INTC','IP','IR',
    'IWM','IYR','JNJ','KDP','KMI','KO','KRE','KWEB','LOW','LRCX','LUV','LVS','LYFT',
    'MAR','MARA','MCHP','MGM','MNST','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR',
    'MU','NCLH','NEE','NEM','NKE','NUE','NVDA','NVO','NXPI','ON','ORCL','OXY','PANW',
    'PCAR','PDD','PEP','PFE','PINS','PLTR','PYPL','QCOM','QQQ','QQQM','RBLX','RIOT',
    'RIVN','RTX','SBUX','SCHW','SHOP','SJM','SLB','SLV','SMCI','SMH','SNAP','SNOW',
    'SOFI','SOXX','SPLG','SPY','TER','TGT','TJX','TLT','TMUS','TQQQ','TSCO','TSLA',
    'TTD','TTWO','TWLO','TXN','U','UAL','UBER','UPS','USB','USO','VLO','VNQ','VRT',
    'VST','VT','VTR','WMT','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP',
    'XLRE','XLU','XLV','XLY','XOM','XOP','XRT'
]
#[ 'AA','AAL','AAOI','AAPL','ABNB','ACHR','ACLX','ACWI','AEM','AFRM','AG','AGX','AI','ALAB','ALB','AMAT','AMD','AMRZ','AMZN','ANET','APA','APH','APLD','APP','ARKK','ARM','ARWR','ASND','ASTS','AVAV','AVGO','BA','BABA','BAC','BE','BKR','BLDR','BMNR','BROS','C','CARR','CAT','CAVA','CCJ','CCL','CDE','CE','CEG','CELH','CIEN','CIFR','CLF','CLS','CLSK','CMG','CNC','COHR','COIN','COMP','CORZ','CPRT','CRCL','CRDO','CRM','CRWD','CRWV','CSCO','CSX','CVE','CVNA','CVS','CVX','CZR','DAL','DASH','DDOG','DE','DELL','DHR','DIA','DIS','DKNG','DLTR','DOCN','DOW','DUOL','DVN','DXCM','EA','EBAY','EEM','EL','ELF','EMR','ENPH','ENTG','EQT','ETSY','EWJ','EWT','EWW','EWY','EWZ','EXAS','EXC','F','FANG','FCX','FDX','FIG','FIGR','FITB','FN','FROG','FSLR','FSLY','FTAI','FTNT','FTV','FXI','GAP','GBTC','GDX','GDXJ','GEHC','GEV','GFS','GH','GIS','GLW','GOOG','GOOGL','GS','GTLB','HAL','HD','HIMS','HL','HLT','HON','HOOD','HPE','HPQ','HSY','HUT','HWM','HYMC','IAU','IBM','IBRX','IGV','IJH','IJR','INTC','IONQ','IOT','IP','IQV','IR','IREN','IWM','IYR','JCI','JNJ','JOBY','JPM','KDP','KLAC','KMI','KO','KRE','KRMN','KTOS','KWEB','LEU','LITE','LLY','LMND','LMT','LOW','LRCX','LSCC','LUNR','LUV','LVS','LYFT','MA','MAR','MARA','MCHP','MCK','MDGL','MDLN','MELI','META','MGM','MKSI','MNDY','MNST','MOD','MP','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR','MU','NBIS','NCLH','NEE','NEM','NET','NFLX','NGD','NKE','NU','NUE','NVDA','NVO','NXPI','NXT','OKLO','ON','ONDS','ONON','ONTO','OPEN','ORCL','OVV','OXY','PANW','PATH','PCAR','PDD','PEP','PFE','PH','PINS','PL','PLTR','PM','PSKY','PSTG','PYPL','Q','QBTS','QCOM','QQQ','QQQM','QXO','RBLX','RBRK','RCL','RDDT','RGTI','RH','RIG','RIOT','RIVN','RKLB','RKT','RMBS','ROKU','RTX','RUN','RVMD','SAIA','SATS','SBUX','SCHW','SE','SFM','SHOP','SJM','SLB','SLV','SM','SMCI','SMH','SMR','SN','SNAP','SNDK','SNOW','SOFI','SOLS','SOUN','SOXX','SPLG','SPOT','SPY','STRL','STX','SYK','TEAM','TEM','TER','TGT','TJX','TLN','TLT','TMUS','TOST','TQQQ','TSCO','TSEM','TSLA','TSM','TTD','TTMI','TTWO','TWLO','TXN','U','UAL','UBER','UPS','UPST','USAR','USB','USO','UUUU','VAL','VG','VLO','VNQ','VRT','VST','VT','VTR','VTRS','W','WBD','WDC','WFC','WIX','WMB','WMT','WULF','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY','XOM','XOP','XRT','Z','ZS']

# ==============================================================================
# ### ENHANCED NUMBA KERNELS (CORRECTED)
# ==============================================================================

@jit(nopython=True, cache=True)
def _lin_slope_nb(y):
    n = len(y)
    if n < 2: return 0.0
    x_mean = (n - 1) / 2.0
    y_mean = 0.0
    for i in range(n): y_mean += y[i]
    y_mean /= n
    num = 0.0; den = 0.0
    for i in range(n):
        dx = i - x_mean
        num += dx * (y[i] - y_mean)
        den += dx * dx
    return num / den if den != 0.0 else 0.0

@jit(nopython=True, cache=True)
def _rolling_linslope(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _lin_slope_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _lin_fitted_nb(y):
    n = len(y)
    if n < 2: return y[-1] if len(y) > 0 else 0.0
    x_mean = (n - 1) / 2.0
    y_mean = 0.0
    for i in range(n): y_mean += y[i]
    y_mean /= n
    num = 0.0; den = 0.0
    for i in range(n):
        dx = i - x_mean
        num += dx * (y[i] - y_mean)
        den += dx * dx
    slope = num / den if den != 0.0 else 0.0
    return y_mean + slope * ((n-1) - x_mean)

@jit(nopython=True, cache=True)
def _rolling_linreg_fitted(arr, window):
    n = len(arr); out = np.zeros(n)
    for i in range(window - 1, n):
        out[i] = _lin_fitted_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _hurst_nb(y):
    n = len(y)
    if n < 2: return 0.5
    mean = 0.0
    for i in range(n): mean += y[i]
    mean /= n
    var = 0.0
    for i in range(n): var += (y[i] - mean) ** 2
    std = (var / n) ** 0.5
    if std < 1e-12: return 0.5
    r = np.log(std + 1e-9) / np.log(n)
    return r if not np.isnan(r) else 0.5

@jit(nopython=True, cache=True)
def _rolling_hurst(arr, window):
    n = len(arr); out = np.full(n, 0.5)
    for i in range(window - 1, n):
        out[i] = _hurst_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _cog_nb(y):
    n = len(y)
    if n < 2: return 0.0
    num = 0.0; den = 0.0
    for i in range(n):
        w = float(i + 1)
        num += w * y[i]
        den += y[i]
    return -num / (den + 1e-9)

@jit(nopython=True, cache=True)
def _rolling_cog(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _cog_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _shannon_nb(y, bins=10):
    n = len(y)
    if n < 2: return 0.0
    mn = y[0]; mx = y[0]
    for i in range(1, n):
        if y[i] < mn: mn = y[i]
        if y[i] > mx: mx = y[i]
    if mx == mn: return 0.0
    counts = np.zeros(bins)
    for i in range(n):
        idx = int((y[i] - mn) / (mx - mn) * bins)
        if idx >= bins: idx = bins - 1
        counts[idx] += 1.0
    entropy = 0.0
    for i in range(bins):
        p = counts[i] / n + 1e-9
        entropy -= p * np.log(p)
    return entropy

@jit(nopython=True, cache=True)
def _rolling_shannon(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _shannon_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _r_sq_nb(y):
    n = len(y)
    if n < 2: return 0.0
    x_mean = (n - 1) / 2.0
    y_mean = 0.0
    for i in range(n): y_mean += y[i]
    y_mean /= n
    num = 0.0; den_x = 0.0; den_y = 0.0
    for i in range(n):
        dx = i - x_mean; dy = y[i] - y_mean
        num   += dx * dy
        den_x += dx * dx
        den_y += dy * dy
    if den_x == 0.0 or den_y == 0.0: return 0.0
    r = num / ((den_x ** 0.5) * (den_y ** 0.5))
    return r * r

@jit(nopython=True, cache=True)
def _rolling_r_sq(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    for i in range(window - 1, n):
        out[i] = _r_sq_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _rolling_wma(arr, window):
    n = len(arr); out = np.full(n, np.nan)
    w_sum = window * (window + 1) / 2.0
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += arr[i - window + 1 + j] * (j + 1)
        out[i] = s / w_sum
    return out

@jit(nopython=True, cache=True)
def _kalman_numba(price, r=0.0001, q=0.001):
    x_hat = np.zeros_like(price); p = np.zeros_like(price)
    x_hat[0] = price[0]; p[0] = 1.0
    for t in range(1, len(price)):
        p_minus  = p[t-1] + q
        k        = p_minus / (p_minus + r)
        x_hat[t] = x_hat[t-1] + k * (price[t] - x_hat[t-1])
        p[t]     = (1 - k) * p_minus
    return x_hat

@jit(nopython=True, cache=True)
def _rsi_nb(arr, period=14):
    n = len(arr); rsi = np.full(n, 50.0)
    gains = np.zeros(n); losses = np.zeros(n)

    for i in range(1, n):
        delta = arr[i] - arr[i-1]
        gains[i] = max(delta, 0.0)
        losses[i] = max(-delta, 0.0)

    if period >= n: return rsi

    avg_gain = np.mean(gains[1:period+1])
    avg_loss = np.mean(losses[1:period+1])

    for i in range(period, n):
        avg_gain = (avg_gain * (period-1) + gains[i]) / period
        avg_loss = (avg_loss * (period-1) + losses[i]) / period
        rs = avg_gain / (avg_loss + 1e-9)
        rsi[i] = 100.0 - (100.0 / (1.0 + rs))
    return rsi

@jit(nopython=True, cache=True)
def _fisher_transform_nb(arr, period=10):
    n = len(arr)
    fisher = np.zeros(n)
    value = np.zeros(n)

    if period >= n: return fisher

    for i in range(period-1, n):
        window = arr[i-period+1:i+1]
        min_val = np.min(window)
        max_val = np.max(window)

        if max_val != min_val:
            value[i] = 2 * ((arr[i] - min_val) / (max_val - min_val) - 0.5)

        value[i] = max(min(value[i], 0.999), -0.999)
        fisher[i] = 0.5 * np.log((1 + value[i]) / (1 - value[i]))

        if i > period:
            fisher[i] = 0.5 * fisher[i] + 0.5 * fisher[i-1]

    return fisher

@jit(nopython=True, cache=True)
def _psar_nb(high, low, close, af_start=0.02, af_max=0.2):
    n = len(close)
    psar = np.zeros(n)
    trend = np.ones(n)

    if n < 2: return psar, trend

    psar[0] = low[0]
    af = af_start
    ep = high[0]

    for i in range(1, n):
        psar[i] = psar[i-1] + af * (ep - psar[i-1])

        if trend[i-1] == 1:
            psar[i] = min(psar[i], low[i-1], low[i-2] if i>1 else low[i-1])
            if low[i] < psar[i]:
                trend[i] = -1
                psar[i] = ep
                ep = low[i]
                af = af_start
            else:
                trend[i] = 1
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_start, af_max)
        else:
            psar[i] = max(psar[i], high[i-1], high[i-2] if i>1 else high[i-1])
            if high[i] > psar[i]:
                trend[i] = 1
                psar[i] = ep
                ep = high[i]
                af = af_start
            else:
                trend[i] = -1
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_start, af_max)

    return psar, trend

@jit(nopython=True, cache=True)
def _vhf_nb(price, window=28):
    n = len(price)
    vhf = np.zeros(n)

    if window >= n: return vhf

    for i in range(window, n):
        segment = price[i-window+1:i+1] # FIX: Correctly aligns up to day i
        hcp = np.max(segment) - np.min(segment)
        sum_changes = 0.0
        for j in range(1, window):
            sum_changes += abs(segment[j] - segment[j-1])
        vhf[i] = hcp / (sum_changes + 1e-9)

    return vhf

@jit(nopython=True, cache=True)
def _choppiness_nb(high, low, close, period=14):
    n = len(close)
    chop = np.full(n, 50.0)

    if period >= n: return chop

    for i in range(period, n):
        atr_sum = 0.0
        for j in range(i-period+1, i+1):
            tr = max(high[j]-low[j],
                    abs(high[j]-close[j-1]) if j>0 else 0,
                    abs(low[j]-close[j-1]) if j>0 else 0)
            atr_sum += tr

        max_high = np.max(high[i-period+1:i+1])
        min_low = np.min(low[i-period+1:i+1])

        chop[i] = 100 * np.log10(atr_sum / (max_high - min_low + 1e-9)) / np.log10(period)

    return chop

@jit(nopython=True, cache=True)
def _kama_nb(price, er, fast=2, slow=30):
    n = len(price)
    kama = np.zeros(n)
    kama[0] = price[0]
    fast_sc = 2.0 / (fast + 1)
    slow_sc = 2.0 / (slow + 1)
    for i in range(1, n):
        er_val = er[i] if not np.isnan(er[i]) else 0.0  # NaN → slowest adaptation
        sc = (er_val * (fast_sc - slow_sc) + slow_sc) ** 2
        kama[i] = kama[i-1] + sc * (price[i] - kama[i-1])
    return kama

@jit(nopython=True, cache=True)
def _fractal_energy_nb(y):
    n = len(y)
    if n < 3: return 0.0
    energy = 0.0
    for i in range(1, n-1):
        d2 = (y[i+1] - 2*y[i] + y[i-1])
        energy += d2 * d2
    return np.sqrt(energy / (n-2))

@jit(nopython=True, cache=True)
def _rolling_fractal_energy(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    if window >= n: return out
    for i in range(window - 1, n):
        out[i] = _fractal_energy_nb(arr[i - window + 1 : i + 1])
    return out

@jit(nopython=True, cache=True)
def _aroon_up_nb(high, window):
    n = len(high)
    aroon = np.zeros(n)

    if window >= n: return aroon

    for i in range(window, n):
        window_high = high[i-window+1:i+1] # FIX: Correctly aligns up to day i
        days_since = window - 1 - np.argmax(window_high)
        aroon[i] = (window - days_since) / window
    return aroon

def _warm_up_numba():
    dummy = np.random.randn(60).astype(np.float64)
    _rolling_linslope(dummy, 10)
    _rolling_linreg_fitted(dummy, 10)
    _rolling_hurst(dummy, 50)
    _rolling_cog(dummy, 20)
    _rolling_shannon(dummy, 20)
    _rolling_r_sq(dummy, 30)
    _rolling_wma(dummy, 10)
    _kalman_numba(dummy)
    _rsi_nb(dummy, 14)
    _fisher_transform_nb(dummy, 10)
    _psar_nb(dummy, dummy, dummy)
    _vhf_nb(dummy, 28)
    _choppiness_nb(dummy, dummy, dummy, 14)
    _rolling_fractal_energy(dummy, 20)
    _aroon_up_nb(dummy, 25)
    print("✅ Numba kernels compiled and ready\n")

_warm_up_numba()

# ==============================================================================
# ### DIAGNOSTICS
# ==============================================================================

def diagnose_nans(df, feature_cols):
    nan_counts = {}
    for col in feature_cols:
        if col in df.columns:
            nan_count = df[col].isna().sum()
            if nan_count > 0:
                nan_counts[col] = nan_count

    if nan_counts:
        print("\n🔍 NaN DIAGNOSIS (Top 10 worst offenders):")
        sorted_nans = sorted(nan_counts.items(), key=lambda x: x[1], reverse=True)[:10]
        for col, count in sorted_nans:
            pct = (count / len(df)) * 100
            print(f"  {col}: {count:,} NaNs ({pct:.1f}%)")
    return nan_counts

# ==============================================================================
# ### FEATURE FACTORY v3.20.2 — CLEANED UP (CORRECTED)
# ==============================================================================

def generate_factory_features_v2(df):
    df = df.copy()
    df['hlc3'] = (df['high'] + df['low'] + df['close']) / 3

    hlc = df['hlc3'].values.astype(np.float64)
    hi  = df['high'].values.astype(np.float64)
    lo  = df['low'].values.astype(np.float64)
    cl  = df['close'].values.astype(np.float64)
    vol = df['volume'].values.astype(np.float64)
    idx = df.index

    print(f"  [DEBUG] Input data length: {len(df)} rows")

    # ══════════════════════════════════════════════════════════════════════
    # TREND & SMOOTHNESS
    # ══════════════════════════════════════════════════════════════════════

    ema30 = pd.Series(hlc, index=idx).ewm(span=30).mean().values
    ema30_2 = pd.Series(ema30, index=idx).ewm(span=30).mean().values
    ema30_3 = pd.Series(ema30_2, index=idx).ewm(span=30).mean().values
    tema_30 = 3*ema30 - 3*ema30_2 + ema30_3
    tema_30_pct = hlc / (tema_30 + 1e-9) - 1

    sma_20 = pd.Series(hlc, index=idx).rolling(20).mean().values
    sma_20_pct = hlc / (sma_20 + 1e-9) - 1

    wma1 = _rolling_wma(hlc, 10)
    wma2 = _rolling_wma(hlc, 21)
    hma_raw = 2 * wma1 - wma2
    hma_21 = pd.Series(hma_raw, index=idx).rolling(5).mean().values
    hma_21_pct = hlc / (hma_21 + 1e-9) - 1

    kalman = _kalman_numba(hlc)
    kalman_pct = hlc / (kalman + 1e-9) - 1

    hlc_s = pd.Series(hlc, index=idx)
    er_20 = (hlc_s.diff(20).abs() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values
    er_10 = (hlc_s.diff(10).abs() / (hlc_s.diff().abs().rolling(10).sum() + 1e-9)).values

    kama_20 = _kama_nb(hlc, er_20)
    kama_20_pct = hlc / (kama_20 + 1e-9) - 1

    linreg_30 = _rolling_linslope(hlc, 30)
    r_sq_30 = _rolling_r_sq(hlc, 30)
    r_sq_10 = _rolling_r_sq(hlc, 10)

    # FIX: Rolling standard deviation stops global volatility leakage
    slope_std = pd.Series(linreg_30, index=idx).rolling(30).std().values + 1e-9
    logistic_prob_30 = 1.0 / (1.0 + np.exp(-linreg_30 / slope_std))

    # ══════════════════════════════════════════════════════════════════════
    # EFFICIENCY & STRENGTH
    # ══════════════════════════════════════════════════════════════════════

    cl_prev = np.roll(cl, 1); cl_prev[0] = cl[0]
    tr = np.maximum(hi-lo, np.maximum(np.abs(hi-cl_prev), np.abs(lo-cl_prev)))
    atr_14 = pd.Series(tr, index=idx).rolling(14).mean().values

    hi_prev = np.roll(hi, 1); hi_prev[0] = hi[0]
    lo_prev = np.roll(lo, 1); lo_prev[0] = lo[0]
    plus_dm = np.where((hi-hi_prev) > (lo_prev-lo), np.maximum(hi-hi_prev, 0), 0).astype(np.float64)
    minus_dm = np.where((lo_prev-lo) > (hi-hi_prev), np.maximum(lo_prev-lo, 0), 0).astype(np.float64)

    pdi14 = 100 * (pd.Series(plus_dm,index=idx).rolling(14).mean() / (atr_14 + 1e-9))
    mdi14 = 100 * (pd.Series(minus_dm,index=idx).rolling(14).mean() / (atr_14 + 1e-9))
    adx_14 = (100 * np.abs(pdi14-mdi14) / (pdi14+mdi14+1e-9)).rolling(14).mean().values

    pdi_14 = pdi14.values
    mdi_14 = mdi14.values
    di_spread = pdi_14 - mdi_14

    vhf_28 = _vhf_nb(hlc, 28)

    # ══════════════════════════════════════════════════════════════════════
    # PERSISTENCE & CHAOS
    # ══════════════════════════════════════════════════════════════════════

    hurst_50 = _rolling_hurst(hlc, 50)
    hurst_20 = _rolling_hurst(hlc, 20)

    shannon_20 = _rolling_shannon(hlc, 20)
    shannon_10 = _rolling_shannon(hlc, 10)

    fractal_energy_20 = _rolling_fractal_energy(hlc, 20)

    sma_30 = pd.Series(hlc, index=idx).rolling(30).mean().values
    d_sma = hlc / (sma_30 + 1e-9) - 1
    d_tema = hlc / (tema_30 + 1e-9) - 1
    d_kal = hlc / (kalman + 1e-9) - 1
    dispersion_30 = np.std(np.stack([d_sma, d_tema, d_kal], axis=1), axis=1)

    # ══════════════════════════════════════════════════════════════════════
    # BOUNDED OSCILLATORS
    # ══════════════════════════════════════════════════════════════════════

    rsi_14 = _rsi_nb(hlc, 14)
    choppiness_14 = _choppiness_nb(hi, lo, cl, 14)
    vidya_cmo_20 = (hlc_s.diff().rolling(20).sum() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values

    # ══════════════════════════════════════════════════════════════════════
    # POSITION & TIMING
    # ══════════════════════════════════════════════════════════════════════

    hi_s = pd.Series(hi, index=idx)
    donchian_high_50 = (hi_s / hi_s.rolling(50).max() - 1).values
    aroon_up_25 = _aroon_up_nb(hi, 25)

    lr_close_10 = _rolling_linreg_fitted(cl, 10)
    lr_close_10[:10] = cl[:10]

    psar_values, psar_trend = _psar_nb(hi, lo, lr_close_10)
    psar_distance = (cl - psar_values) / (psar_values + 1e-9)

    cog_20 = _rolling_cog(hlc, 20)

    # ══════════════════════════════════════════════════════════════════════
    # VOLUME-WEIGHTED
    # ══════════════════════════════════════════════════════════════════════

    _tp_v = pd.Series(hlc * vol, index=idx)
    _vol_s = pd.Series(vol, index=idx)
    _cl_s = pd.Series(cl, index=idx)
    mtsi = (_cl_s - (_tp_v.rolling(2).sum() / (_vol_s.rolling(2).sum() + 1e-9))).ewm(span=3).mean().values

    fisher_10 = _fisher_transform_nb(hlc, 10)

    # ══════════════════════════════════════════════════════════════════════
    # RATIO INDICATORS
    # ══════════════════════════════════════════════════════════════════════

    LN10 = np.log(10)

    hurst_shannon_product = hurst_50 * (1 - shannon_20/LN10)
    shannon_hurst_delta = shannon_20/LN10 - hurst_50
    entropy_r_sq_ratio = shannon_20 / (r_sq_30 + 1e-9)
    chaos_score = (shannon_20/LN10) * (1 - hurst_50) * dispersion_30
    shannon_ratio_10_20 = shannon_10 / (shannon_20 + 1e-9)

    ea_adx = (adx_14 / 100.0) * er_20
    hurst_er_divergence = hurst_50 - er_20
    adx_slope = _rolling_linslope(adx_14, 10)
    trend_purity = r_sq_30 * er_20 * (1 - shannon_20/LN10) * (adx_14/100.0)
    er_slope = _rolling_linslope(er_20, 10)

    rsi_fisher_spread = (rsi_14 - 50) / 50.0 - fisher_10

    tema_signal = np.sign(tema_30_pct)
    sma_signal = np.sign(sma_20_pct)
    hma_signal = np.sign(hma_21_pct)
    kalman_signal = np.sign(kalman_pct)
    kama_signal = np.sign(kama_20_pct)
    ma_consensus = (tema_signal + sma_signal + hma_signal + kalman_signal + kama_signal) / 5.0

    psar_trend_slope_align = psar_trend * np.sign(linreg_30)
    r_sq_hurst_ratio = r_sq_30 / (hurst_50 + 1e-9)
    cog_mtsi_delta = (cog_20 / 20.0) - mtsi
    di_slope_align = (di_spread / 100.0) * np.sign(linreg_30)
    vhf_adx_ratio = vhf_28 / (adx_14/100.0 + 1e-9)
    dispersion_r_sq = dispersion_30 / (r_sq_30 + 1e-9)
    slope_quality = np.abs(linreg_30) * r_sq_30

    hurst_differential = hurst_50 - hurst_20
    er_ratio_10_20 = er_10 / (er_20 + 1e-9)
    r_sq_ratio_10_30 = r_sq_10 / (r_sq_30 + 1e-9)

    donchian_aroon = (1 + donchian_high_50) * aroon_up_25
    aroon_slope_align = aroon_up_25 * np.sign(linreg_30)
    psar_kalman_delta = psar_distance - kalman_pct

    chop_hurst_ratio = choppiness_14 / (hurst_50*100.0 + 1e-9)
    fractal_shannon_ratio = fractal_energy_20 / (shannon_20 + 1e-9)
    vhf_chop_ratio = vhf_28 / (choppiness_14/100.0 + 1e-9)

    rsi_slope = _rolling_linslope(rsi_14, 10)
    vhf_slope = _rolling_linslope(vhf_28, 10)
    # ══════════════════════════════════════════════════════════════════════
    # Z_LENS_INDICATORS - CORE 16 ONLY (as per original design)
    # ══════════════════════════════════════════════════════════════════════

    Z_LENS_INDICATORS = {
        'er_20':            pd.Series(er_20,            index=idx),
        'vidya_cmo_20':     pd.Series(vidya_cmo_20,     index=idx),
        'r_sq_30':          pd.Series(r_sq_30,          index=idx),
        'hurst_50':         pd.Series(hurst_50,         index=idx),
        'shannon_20':       pd.Series(shannon_20,       index=idx),
        'adx_14':           pd.Series(adx_14,           index=idx),
        'logistic_prob_30': pd.Series(logistic_prob_30, index=idx),
        'aroon_up_25':      pd.Series(aroon_up_25,      index=idx),
        'donchian_high_50': pd.Series(donchian_high_50, index=idx),
        'dispersion_30':    pd.Series(dispersion_30,    index=idx),
        'lr_slope_30':      pd.Series(linreg_30,        index=idx),
        'tema_30_pct':      pd.Series(tema_30_pct,      index=idx),
        'sma_20_pct':       pd.Series(sma_20_pct,       index=idx),
        'hma_21_pct':       pd.Series(hma_21_pct,       index=idx),
        'kalman_pct':       pd.Series(kalman_pct,       index=idx),
        'mtsi':             pd.Series(mtsi,             index=idx),
    }

    # BOUNDED OSCILLATORS → LENS_10 only (z, slope, sos)
    BOUNDED_OSCILLATORS = {
        'rsi_14':         pd.Series(rsi_14,         index=idx),
        'choppiness_14':  pd.Series(choppiness_14,  index=idx),
    }

    # UNBOUNDED INDICATORS → WIN (rolling %)
    UNBOUNDED_INDICATORS = {
        'vhf_28':                 pd.Series(vhf_28,                 index=idx),
        'pdi_14':                 pd.Series(pdi_14,                 index=idx),
        'mdi_14':                 pd.Series(mdi_14,                 index=idx),
        'di_spread':              pd.Series(di_spread,              index=idx),
        'fractal_energy_20':      pd.Series(fractal_energy_20,      index=idx),
        'psar_distance':          pd.Series(psar_distance,          index=idx),
        'psar_trend':             pd.Series(psar_trend,             index=idx),
        'fisher_10':              pd.Series(fisher_10,              index=idx),
        'hurst_shannon_product':  pd.Series(hurst_shannon_product,  index=idx),
        'shannon_hurst_delta':    pd.Series(shannon_hurst_delta,    index=idx),
        'entropy_r_sq_ratio':     pd.Series(entropy_r_sq_ratio,     index=idx),
        'chaos_score':            pd.Series(chaos_score,            index=idx),
        'shannon_ratio_10_20':    pd.Series(shannon_ratio_10_20,    index=idx),
        'ea_adx':                 pd.Series(ea_adx,                 index=idx),
        'hurst_er_divergence':    pd.Series(hurst_er_divergence,    index=idx),
        'adx_slope':              pd.Series(adx_slope,              index=idx),
        'trend_purity':           pd.Series(trend_purity,           index=idx),
        'er_slope':               pd.Series(er_slope,               index=idx),
        'rsi_fisher_spread':      pd.Series(rsi_fisher_spread,      index=idx),
        'psar_trend_slope_align': pd.Series(psar_trend_slope_align, index=idx),
        'r_sq_hurst_ratio':       pd.Series(r_sq_hurst_ratio,       index=idx),
        'cog_mtsi_delta':         pd.Series(cog_mtsi_delta,         index=idx),
        'di_slope_align':         pd.Series(di_slope_align,         index=idx),
        'vhf_adx_ratio':          pd.Series(vhf_adx_ratio,          index=idx),
        'dispersion_r_sq':        pd.Series(dispersion_r_sq,        index=idx),
        'slope_quality':          pd.Series(slope_quality,          index=idx),
        'hurst_differential':     pd.Series(hurst_differential,     index=idx),
        'er_ratio_10_20':         pd.Series(er_ratio_10_20,         index=idx),
        'r_sq_ratio_10_30':       pd.Series(r_sq_ratio_10_30,       index=idx),
        'donchian_aroon':         pd.Series(donchian_aroon,         index=idx),
        'aroon_slope_align':      pd.Series(aroon_slope_align,      index=idx),
        'psar_kalman_delta':      pd.Series(psar_kalman_delta,      index=idx),
        'chop_hurst_ratio':       pd.Series(chop_hurst_ratio,       index=idx),
        'fractal_shannon_ratio':  pd.Series(fractal_shannon_ratio,  index=idx),
        'vhf_chop_ratio':         pd.Series(vhf_chop_ratio,         index=idx),
        'rsi_slope':              pd.Series(rsi_slope,              index=idx),
        'vhf_slope':              pd.Series(vhf_slope,              index=idx),
    }
    # ══════════════════════════════════════════════════════════════════════
    # APPLY TRANSFORMATIONS
    # ══════════════════════════════════════════════════════════════════════

    for name, ind in Z_LENS_INDICATORS.items():
        arr = ind.values.astype(np.float64)
        for lens in [10, 90]:
            rm = pd.Series(arr, index=idx).rolling(lens).mean().values
            rs = pd.Series(arr, index=idx).rolling(lens).std().values
            z = (arr - rm) / (rs + 1e-9)
            zs = _rolling_linslope(z, lens)
            zsos = _rolling_linslope(zs, lens)
            df[f'LENS_{lens}_{name}_z'] = z
            df[f'LENS_{lens}_{name}_z_slope'] = zs
            df[f'LENS_{lens}_{name}_z_sos'] = zsos

    for name, ind in BOUNDED_OSCILLATORS.items():
        arr = ind.values.astype(np.float64)
        for lens in [10]:
            rm = pd.Series(arr, index=idx).rolling(lens).mean().values
            rs = pd.Series(arr, index=idx).rolling(lens).std().values
            z = (arr - rm) / (rs + 1e-9)
            zs = _rolling_linslope(z, lens)
            zsos = _rolling_linslope(zs, lens)
            df[f'LENS_{lens}_{name}_z'] = z
            df[f'LENS_{lens}_{name}_z_slope'] = zs
            df[f'LENS_{lens}_{name}_z_sos'] = zsos

    for name, ind in UNBOUNDED_INDICATORS.items():
        arr = ind.values.astype(np.float64)
        for win in [10, 30]:
            rm = pd.Series(arr, index=idx).rolling(win).mean().values
            df[f'WIN_{win}_{name}_pct'] = arr / (rm + 1e-9) - 1

    cog_arr = cog_20.astype(np.float64)
    for win in [10, 30, 90]:
        rm = pd.Series(cog_arr, index=idx).rolling(win).mean().values
        df[f'WIN_{win}_cog_20_pct'] = cog_arr / (rm + 1e-9) - 1

    # ══════════════════════════════════════════════════════════════════════
    # CLEANUP & TARGET CREATION
    # ══════════════════════════════════════════════════════════════════════

    df = df.replace([np.inf, -np.inf], np.nan)

    # FIX: Ensure final day gets NaN so dropna works accurately
    target = df['close'].shift(-1) > df['close']
    df['T_FINAL'] = target.astype(float)
    df.loc[df.index[-1], 'T_FINAL'] = np.nan

    feature_cols = [c for c in df.columns if c.startswith('LENS_') or c.startswith('WIN_')]

    initial_len = len(df)
    nan_counts = diagnose_nans(df, feature_cols + ['T_FINAL'])

    df = df.dropna(subset=feature_cols + ['T_FINAL'])
    dropped = initial_len - len(df)

    if len(df) == 0:
        print(f"  🚨 ALL {initial_len} ROWS DROPPED!")
        return pd.DataFrame()

    z_lens_count = len(Z_LENS_INDICATORS) * 2 * 3
    bounded_count = len(BOUNDED_OSCILLATORS) * 1 * 3
    unbounded_count = len(UNBOUNDED_INDICATORS) * 2
    cog_count = 3

    total_features = z_lens_count + bounded_count + unbounded_count + cog_count

    print(f"  [FEATURES] {total_features} total ({z_lens_count} Z_LENS + {bounded_count} BOUNDED + {unbounded_count} UNBOUNDED + {cog_count} COG)")
    print(f"  [CLEANUP] Dropped {dropped} rows containing NaNs. Remaining valid rows: {len(df)}\n")
    return df

# ==============================================================================
# ### DATA LOADING - HELPER FUNCTIONS (ADD THIS BEFORE load_hybrid_data_parallel)
# ==============================================================================
def _fetch_symbol(sym_period):
    """Helper for parallel downloads"""
    sym, period = sym_period
    try:
        df = yf.download(sym, period=period, progress=False)

        if df.empty:
            return None

        # FIX: Flatten multi-index columns (yfinance quirk)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.columns = [c.lower() for c in df.columns]

        # Verify required columns
        required = ['open', 'high', 'low', 'close', 'volume']
        missing = [c for c in required if c not in df.columns]
        if missing:
            return None

        # NEW: Reject symbols with insufficient history
        MIN_ROWS = 400  # ~2 years minimum for 6y request
        if len(df) < MIN_ROWS:
            print(f"  ⚠️  {sym}: Only {len(df)} rows (need {MIN_ROWS}+)")
            return None

        return (sym, df)

    except Exception as e:
        return None

def _process_symbol_features(item):
    """Helper for parallel feature generation"""
    sym, df = item
    try:
        result = generate_factory_features_v2(df)
        return result
    except Exception as e:
        print(f"  ⚠️  {sym} failed: {e}")
        return pd.DataFrame()

# ==============================================================================
# ### DATA LOADING (REPLACE YOUR CURRENT VERSION)
# ==============================================================================
def load_hybrid_data_parallel(brain_name, symbols, period='2y', n_workers=N_FEATURE_WORKERS):
    print(f"📥 Parallel download: {len(symbols)} symbols...")

    # Prepare arguments for parallel download
    download_args = [(sym, period) for sym in symbols]

    with ThreadPoolExecutor(max_workers=min(10, len(symbols))) as executor:
        results = list(tqdm(executor.map(_fetch_symbol, download_args),
                           total=len(symbols), desc="⬇ Downloading"))

    valid = [r for r in results if r is not None]
    print(f"   ✅ {len(valid)}/{len(symbols)} symbols fetched")

    if not valid:
        print("  🚨 NO SYMBOLS DOWNLOADED - check network/API")
        return pd.DataFrame()

    print(f"⚙ Building features in parallel (workers={n_workers})...")

    # Use ThreadPoolExecutor instead of ProcessPoolExecutor (simpler, works in Colab)
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        processed = list(tqdm(executor.map(_process_symbol_features, valid),
                             total=len(valid), desc="⚙ Features"))

    processed = [df for df in processed if not df.empty]

    if not processed:
        print("  🚨 NO FEATURES GENERATED - all symbols failed")
        return pd.DataFrame()

    master = pd.concat(processed, axis=0, ignore_index=True)
    print(f"   ✅ Combined master dataset: {len(master):,} rows\n")
    return master

# ==============================================================================
# ### GPU-ACCELERATED AUDIT
# ==============================================================================

def build_full_model(model_type, n_features, seq_len, device=DEVICE):
    """Simplified model for next-day prediction"""
    with tf.device(device):
        model = Sequential([
            Input(shape=(seq_len, n_features)),
            GRU(64, return_sequences=True) if model_type == 'GRU' else LSTM(64, return_sequences=True),
            Dropout(0.1),
            GRU(32) if model_type == 'GRU' else LSTM(32),
            Dropout(0.1),
            Dense(16, activation='relu'),
            Dense(1, activation='sigmoid', dtype='float32')
        ])
        model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model

@tf.function
def _eval_accuracy(model, X_batch, y_batch):
    preds = tf.squeeze(model(X_batch, training=False), axis=-1)
    correct = tf.equal(tf.cast(preds >= 0.5, tf.int32), tf.cast(y_batch, tf.int32))
    return tf.reduce_mean(tf.cast(correct, tf.float32))

def run_judicial_audit(brain_name, master_df, model_type='GRU', seq_len=5, epochs=75, batch_size=1024):
    """Walk-forward validation with anti-leakage measures"""
    feature_cols = [c for c in master_df.columns if c.startswith('LENS_') or c.startswith('WIN_')]
    n_features = len(feature_cols)

    print(f"  [FEATURES] Using {n_features} features (LENS + WIN transforms)")

    X_raw = master_df[feature_cols].values.astype(np.float32)
    y_raw = master_df['T_FINAL'].values.astype(np.float32)
    n = len(X_raw)
    X_seqs = np.stack([X_raw[i-seq_len:i] for i in range(seq_len, n)])
    y_seqs = y_raw[seq_len:]

    train_end = int(len(X_seqs) * 0.70)
    val_end = int(len(X_seqs) * 0.85)

    X_tr = X_seqs[:train_end]
    y_tr = y_seqs[:train_end]
    X_val = X_seqs[train_end:val_end]
    y_val = y_seqs[train_end:val_end]
    X_test = X_seqs[val_end:]
    y_test = y_seqs[val_end:]

    scaler = RobustScaler()
    n_train, seq, feats = X_tr.shape
    X_tr_2d = X_tr.reshape(-1, feats)
    scaler.fit(X_tr_2d)

    X_tr_scaled = scaler.transform(X_tr_2d).reshape(n_train, seq, feats)
    X_val_scaled = scaler.transform(X_val.reshape(-1, feats)).reshape(len(X_val), seq, feats)
    X_test_scaled = scaler.transform(X_test.reshape(-1, feats)).reshape(len(X_test), seq, feats)

    print(f"  [DATA] train={len(X_tr):,}  val={len(X_val):,}  test={len(X_test):,}  features={n_features}")
    print(f"  [WALK-FORWARD] Train=70%, Val=15%, Test=15%")
    print(f"  [SCALING] Fitted on TRAIN ONLY")

    AUTO = tf.data.AUTOTUNE
    train_ds = (tf.data.Dataset.from_tensor_slices((X_tr_scaled, y_tr))
                .shuffle(min(20_000, len(X_tr)), reshuffle_each_iteration=True)
                .batch(batch_size).prefetch(AUTO))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val_scaled, y_val))
              .batch(batch_size * 2).prefetch(AUTO))

    model = build_full_model(model_type, n_features, seq_len)

    with tf.device(DEVICE):
        model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=8,
                                           restore_best_weights=True, min_delta=0.0005),
                             ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=3, min_lr=1e-5)],
                  verbose=1)

    X_val_tf = tf.constant(X_val_scaled)
    y_val_tf = tf.constant(y_val)
    X_test_tf = tf.constant(X_test_scaled)
    y_test_tf = tf.constant(y_test)

    val_acc = _eval_accuracy(model, X_val_tf, y_val_tf).numpy()
    test_acc = _eval_accuracy(model, X_test_tf, y_test_tf).numpy()

    print(f"  [MODEL] Validation accuracy: {val_acc:.4f}")
    print(f"  [MODEL] Test accuracy:       {test_acc:.4f}")

    if val_acc > 0.70 and test_acc > 0.70:
        print(f"  🚨 BOTH val and test >70% - SEVERE LEAKAGE")
    elif val_acc > 0.70 and test_acc < 0.65:
        print(f"  ✅ Val high, test normal - MODEL OVERFIT (acceptable)")
    elif test_acc > 0.70:
        print(f"  🚨 Test >70% - LEAKAGE IN UNSEEN DATA")
    else:
        print(f"  ✅ Both scores realistic - NO LEAKAGE DETECTED")

    report_rows = []
    for fi, feat_name in enumerate(tqdm(feature_cols, desc="Permutation scoring")):
        try:
            X_perm = X_val_scaled.copy()
            flat = X_perm[:, :, fi].flatten()
            np.random.shuffle(flat)
            X_perm[:, :, fi] = flat.reshape(X_perm[:, :, fi].shape)
            perm_acc = _eval_accuracy(model, tf.constant(X_perm), y_val_tf).numpy()
            report_rows.append({'Feature': feat_name, 'I_raw': max(0.0, val_acc - perm_acc)})
        except Exception:
            report_rows.append({'Feature': feat_name, 'I_raw': 0.0})

    del model; gc.collect(); tf.keras.backend.clear_session()
    return pd.DataFrame(report_rows), val_acc, test_acc

# ==============================================================================
# ### SOVEREIGN HUNT & DIVERSITY
# ==============================================================================

BRAIN_LOCKS = {
    'DIRECTION': [],
    'EASE': [],
    'EXP': []
}

DATA_WINDOW = {
    'DIRECTION': '6y',
    'EASE': '4y',
    'EXP': '4y',
}

def _parse_feature_name(f):
    TRANSFORM_TOKENS = {'z', 'slope', 'sos', 'pct'}
    if f.startswith('LENS_') or f.startswith('WIN_'):
        parts = f.split('_')
        prefix = parts[0]
        window = parts[1]
        remainder = list(parts[2:])
        while remainder and remainder[-1] in TRANSFORM_TOKENS:
            remainder.pop()
        indicator = '_'.join(remainder)
        family = '_'.join(p for p in remainder if not p.isdigit())
        lookback = f'{prefix}_{window}_{indicator}'
        return prefix, window, lookback, family
    return None, None, f, f

def apply_sovereign_hunt(ledger_df, master_data_df, brain_name, max_slots=19):
    locked_list = BRAIN_LOCKS.get(brain_name, [])
    candidates = ledger_df.sort_values(by='I_raw', ascending=False)
    picked = [f for f in locked_list if f in ledger_df['Feature'].values]

    for lf in locked_list:
        if lf not in ledger_df['Feature'].values:
            print(f"  ⚠️  BRAIN_LOCK '{lf}' not found in feature columns")

    CORR_THRESHOLD = 0.85
    family_lookback = {}

    for f in picked:
        _, _, lookback, family = _parse_feature_name(f)
        if family not in family_lookback:
            family_lookback[family] = lookback

    feat_cols = [c for c in master_data_df.columns if c.startswith('LENS_') or c.startswith('WIN_')]
    corr_matrix = master_data_df[feat_cols].corr()

    for _, row in candidates.iterrows():
        if len(picked) >= max_slots: break
        f_name = row['Feature']
        if f_name in picked: continue

        _, _, f_lookback, f_family = _parse_feature_name(f_name)

        if f_family in family_lookback and family_lookback[f_family] != f_lookback:
            continue

        if len(picked) > 0 and corr_matrix[f_name].loc[picked].max() > CORR_THRESHOLD:
            continue

        picked.append(f_name)
        if f_family not in family_lookback:
            family_lookback[f_family] = f_lookback

    pca = PCA()
    pca.fit(RobustScaler().fit_transform(master_data_df[picked]))
    return picked, np.cumsum(pca.explained_variance_ratio_)

def generate_judicial_ledger(brain_name, report_df, master_data_df, iteration=1):
    df = report_df.copy()
    df['I_Norm'] = (df['I_raw'] - df['I_raw'].min()) / (df['I_raw'].max() - df['I_raw'].min() + 1e-9)

    active_picks, var_map = apply_sovereign_hunt(df, master_data_df, brain_name)
    corr_sub = master_data_df[active_picks].corr().abs()
    avg_corr = (corr_sub.sum().sum() - len(active_picks)) / (len(active_picks)**2 - len(active_picks) + 1e-9)

    print(f"\n╔══ {brain_name} SOVEREIGN CORE v3.20.2 (Iter {iteration}) ══╗")
    print(f"║ {'RNK':<3} | {'FEATURE':<35} | {'UV%':<4} | {'mR':<4} | {'IMPACT':<8} ║")
    print("╠" + "═"*4 + "╬" + "═"*37 + "╬" + "═"*6 + "╬" + "═"*6 + "╬" + "═"*10 + "╣")

    for i, f_name in enumerate(active_picks):
        f_row = df[df['Feature'] == f_name].iloc[0]
        is_locked = f_name in BRAIN_LOCKS.get(brain_name, [])
        icon = "🔒" if is_locked else "🔭"
        lb_val = f_name.split('_')[1] if (f_name.startswith('LENS_') or f_name.startswith('WIN_')) else "??"
        other_picks = [p for p in active_picks if p != f_name]
        max_r = corr_sub[f_name].loc[other_picks].max() if other_picks else 0.0
        uv_val = (1 - corr_sub[f_name].loc[other_picks].mean()) * 100 if other_picks else 100.0

        print(f"║ {i+1:02d}  | {icon} {f_name[:33]:<33} | {uv_val:>3.0f}% | {max_r:.2f} | {f_row['I_Norm']:.4f} ║")
        df.loc[df['Feature'] == f_name, ['UV%','Max_R','LB','Is_Locked']] = [uv_val, max_r, lb_val, is_locked]

    total_var = var_map[-1] if len(var_map) > 0 else 0
    print("╠" + "═"*73 + "╣")
    print(f"║ PCA TOTAL VARIANCE RETENTION: {total_var*100:>33.2f}% ║")
    print(f"║ AVG TEAM CROSS-CORRELATION: {avg_corr:>35.3f} ║")
    print(f"║ SLOTS FILLED: {len(active_picks):>44}/19 ║")
    print("╚" + "═"*73 + "╝")

    return df[df['Feature'].isin(active_picks)]

# ==============================================================================
# ### COMMAND CENTER
# ==============================================================================

if __name__ == '__main__':
    print("\n" + "="*70)
    print("   SOVEREIGN TITAN v3.20.2 — CORE INDICATORS VERIFIED")
    print("="*70)
    print(f"🛡️  Anti-leakage measures active")
    print(f"💾  Results save to: {OUTPUT_DIR}")
    print(f"🌍  Environment: {ENV}\n")

    choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
    BRAINS_TO_RUN = ['DIRECTION','EASE','EXP'] if choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[choice]]

    num_symbols = int(input("Symbols per iteration (Default 50): ") or "50")
    num_iters = int(input("Iterations to run (Default 25): ") or "25")

    final_report_accumulator = []

    for BRAIN in BRAINS_TO_RUN:
        CURRENT_MODEL_TYPE = 'GRU' if BRAIN == 'DIRECTION' else 'LSTM'
        BRAIN_WINDOW = DATA_WINDOW.get(BRAIN, '2y')

        print(f"\n[SYSTEM] Brain: {BRAIN} | Model: {CURRENT_MODEL_TYPE} | Window: {BRAIN_WINDOW} | Device: {DEVICE}")

        for it in range(1, num_iters + 1):
            print(f"\n{'─'*55}")
            print(f"  Iteration {it}/{num_iters}  —  Brain: {BRAIN}")
            print(f"{'─'*55}")

            POOL = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
            master_df = load_hybrid_data_parallel(BRAIN, POOL, period=BRAIN_WINDOW)

            if master_df.empty:
                print("  ⚠️  Empty master_df — skipping.")
                continue

            report_raw, val_acc, test_acc = run_judicial_audit(BRAIN, master_df, model_type=CURRENT_MODEL_TYPE)

            MIN_QUALITY = 0.52
            MAX_QUALITY = 0.70

            if val_acc < MIN_QUALITY:
                print(f"  ⚠️  ITER {it} REJECTED — val_acc {val_acc:.4f} < {MIN_QUALITY}")
                gc.collect(); tf.keras.backend.clear_session()
                continue

            if test_acc > MAX_QUALITY:
                print(f"  🚨 ITER {it} LEAKAGE CONFIRMED — test_acc {test_acc:.4f} > {MAX_QUALITY}")
            elif val_acc > MAX_QUALITY and test_acc < 0.65:
                print(f"  ✅ ITER {it} ACCEPTABLE — val high, test normal (overfit, not leakage)")

            iteration_ledger = generate_judicial_ledger(BRAIN, report_raw, master_df, iteration=it)
            iteration_ledger['Iteration'] = it
            iteration_ledger['Brain'] = BRAIN
            iteration_ledger['Model_Type'] = CURRENT_MODEL_TYPE
            iteration_ledger['Data_Window'] = BRAIN_WINDOW
            iteration_ledger['Val_Acc'] = val_acc
            iteration_ledger['Test_Acc'] = test_acc
            iteration_ledger['Timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            final_report_accumulator.append(iteration_ledger)

            gc.collect()
            tf.keras.backend.clear_session()

    # ==============================================================================
    # ### FINAL EXPORT & SOVEREIGN SELECTION
    # ==============================================================================

    if final_report_accumulator:
        raw_df = pd.concat(final_report_accumulator, axis=0)
        stats = (raw_df.groupby(['Brain','Feature'])
                 .agg(Persistence=('Feature','count'),
                      A_Impact=('I_Norm','mean'),
                      A_UV=('UV%','mean'))
                 .reset_index())

        final_df = (raw_df.merge(stats, on=['Brain','Feature'], how='left')
                          .sort_values(['Brain','Persistence','A_Impact'], ascending=False))

        report_filename = f"Sovereign_Audit_Master_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        report_path = os.path.join(OUTPUT_DIR, report_filename)
        final_df.to_csv(report_path, index=False)

        print("\n" + "="*65)
        print("✅ GLOBAL AUDIT COMPLETE")
        print(f"📊 DATA ROWS COLLECTED: {len(raw_df)}")
        print(f"📂 CSV SAVED TO:        {report_path}")

        avg_acc_by_brain = raw_df.groupby('Brain')['Test_Acc'].mean()
        print("\n" + "="*65)
        print("🛡️  DATA LEAKAGE CHECK (based on UNSEEN test data):")
        for brain, avg_acc in avg_acc_by_brain.items():
            if avg_acc > 0.70:
                status = "🚨 LIKELY LEAKAGE"
            elif avg_acc > 0.65:
                status = "⚠️  SUSPICIOUSLY HIGH"
            elif avg_acc > 0.55:
                status = "✅ REALISTIC"
            else:
                status = "⚠️  UNDERPERFORMING"
            print(f"  {brain:<12} test_acc={avg_acc:.4f}  {status}")
        print("="*65)

        print("\n" + "═"*65)
        print("🚀 FINAL SOVEREIGN ARRAYS (TOP 19 PER BRAIN)")
        print("═"*65)

        FINAL_SELECTIONS = {}
        quality_iters = (raw_df.groupby('Brain')['Iteration'].nunique().to_dict())

        for brain in BRAINS_TO_RUN:
            q_iters = quality_iters.get(brain, num_iters)
            brain_stats = (stats[stats['Brain'] == brain]
                           .sort_values(['Persistence','A_Impact'], ascending=False))

            family_committed = {}
            final_picks = []

            for _, row in brain_stats.iterrows():
                if len(final_picks) >= 19: break
                _, _, lookback, family = _parse_feature_name(row['Feature'])
                if family in family_committed and family_committed[family] != lookback:
                    continue
                final_picks.append(row)
                if family not in family_committed:
                    family_committed[family] = lookback

            top_19 = pd.DataFrame(final_picks)
            FINAL_SELECTIONS[brain] = top_19['Feature'].tolist()

            avg_q_val = raw_df[raw_df['Brain']==brain]['Val_Acc'].mean() if 'Val_Acc' in raw_df.columns else float('nan')
            avg_q_test = raw_df[raw_df['Brain']==brain]['Test_Acc'].mean() if 'Test_Acc' in raw_df.columns else float('nan')

            print(f"\n💎 FINAL 19 — BRAIN: {brain}  (iters: {q_iters}/{num_iters}  val={avg_q_val:.3f}  test={avg_q_test:.3f})")
            print(f"{'RNK':<3} | {'FEATURE':<38} | {'PERSIST':<8} | {'AVG_IMP':<8}")
            print("─" * 62)

            for i, row in top_19.reset_index(drop=True).iterrows():
                print(f"{i+1:02d}  | {row['Feature']:<38} | {int(row['Persistence']):>2}/{q_iters:<5} | {row['A_Impact']:.4f}")

        for brain, winners in FINAL_SELECTIONS.items():
            BRAIN_LOCKS[brain] = winners

        print("\n" + "═"*65)
        print("✅ FINAL 19 SYNCED TO BRAIN_LOCKS")
        print(f"📂 TOTAL UNIQUE FEATURES LOGGED: {len(stats)}")
        print(f"📁 Results saved to: {OUTPUT_DIR}")
        print("═"*65)
    else:
        print("\n⚠️ [CRITICAL] No data collected. Audit failed.")

[ENVIRONMENT] Running in: COLAB
Mounted at /content/drive
✅ Google Drive mounted
[STORAGE] Base output directory: /content/drive/MyDrive/Sovereign_Titan_Results/

⚠️  No GPU detected — running on CPU (slower but functional)
    💡 Enable GPU: Runtime → Change runtime type → GPU
[SYSTEM] Active compute device: /cpu:0

[SYSTEM] Feature generation workers: 1
[SYSTEM] Total CPU cores: 2
[SYSTEM] Results will be saved to: /content/drive/MyDrive/Sovereign_Titan_Results/Sovereign_Titan_v3.20.2_CoreVerified

✅ Numba kernels compiled and ready


   SOVEREIGN TITAN v3.20.2 — CORE INDICATORS VERIFIED
🛡️  Anti-leakage measures active
💾  Results save to: /content/drive/MyDrive/Sovereign_Titan_Results/Sovereign_Titan_v3.20.2_CoreVerified
🌍  Environment: COLAB

Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 4
Symbols per iteration (Default 50): 66
Iterations to run (Default 25): 20

[SYSTEM] Brain: DIRECTION | Model: GRU | Window: 6y | Device: /cpu:0

──────────────────────────────────────────────────

⬇ Downloading:   0%|          | 0/66 [00:00<?, ?it/s]

   ✅ 66/66 symbols fetched
⚙ Building features in parallel (workers=1)...
  [DEBUG] Input data length: 1507 rows


⚙ Features:   0%|          | 0/66 [00:00<?, ?it/s]


🔍 NaN DIAGNOSIS (Top 10 worst offenders):
  LENS_90_donchian_high_50_z_sos: 227 NaNs (15.1%)
  LENS_90_logistic_prob_30_z_sos: 207 NaNs (13.7%)
  LENS_90_dispersion_30_z_sos: 207 NaNs (13.7%)
  LENS_90_adx_14_z_sos: 204 NaNs (13.5%)
  LENS_90_hma_21_pct_z_sos: 202 NaNs (13.4%)
  LENS_90_er_20_z_sos: 198 NaNs (13.1%)
  LENS_90_vidya_cmo_20_z_sos: 198 NaNs (13.1%)
  LENS_90_sma_20_pct_z_sos: 197 NaNs (13.1%)
  LENS_90_mtsi_z_sos: 179 NaNs (11.9%)
  LENS_90_r_sq_30_z_sos: 178 NaNs (11.8%)
  [FEATURES] 179 total (96 Z_LENS + 6 BOUNDED + 74 UNBOUNDED + 3 COG)
  [CLEANUP] Dropped 317 rows containing NaNs. Remaining valid rows: 1190

  [DEBUG] Input data length: 1507 rows

🔍 NaN DIAGNOSIS (Top 10 worst offenders):
  LENS_90_donchian_high_50_z_sos: 227 NaNs (15.1%)
  LENS_90_logistic_prob_30_z_sos: 207 NaNs (13.7%)
  LENS_90_dispersion_30_z_sos: 207 NaNs (13.7%)
  LENS_90_adx_14_z_sos: 204 NaNs (13.5%)
  LENS_90_hma_21_pct_z_sos: 202 NaNs (13.4%)
  LENS_90_er_20_z_sos: 198 NaNs (13.1%)
  LENS

...

   ✅ 66/66 symbols fetched
⚙ Building features in parallel (workers=1)...
  [DEBUG] Input data length: 1004 rows


⚙ Features:   0%|          | 0/66 [00:00<?, ?it/s]


🔍 NaN DIAGNOSIS (Top 10 worst offenders):
  LENS_90_donchian_high_50_z_sos: 227 NaNs (22.6%)
  LENS_90_logistic_prob_30_z_sos: 207 NaNs (20.6%)
  LENS_90_dispersion_30_z_sos: 207 NaNs (20.6%)
  LENS_90_adx_14_z_sos: 204 NaNs (20.3%)
  LENS_90_hma_21_pct_z_sos: 202 NaNs (20.1%)
  LENS_90_er_20_z_sos: 198 NaNs (19.7%)
  LENS_90_vidya_cmo_20_z_sos: 198 NaNs (19.7%)
  LENS_90_sma_20_pct_z_sos: 197 NaNs (19.6%)
  LENS_90_mtsi_z_sos: 179 NaNs (17.8%)
  LENS_90_r_sq_30_z_sos: 178 NaNs (17.7%)
  [FEATURES] 179 total (96 Z_LENS + 6 BOUNDED + 74 UNBOUNDED + 3 COG)
  [CLEANUP] Dropped 317 rows containing NaNs. Remaining valid rows: 687

  [DEBUG] Input data length: 1004 rows

🔍 NaN DIAGNOSIS (Top 10 worst offenders):
  LENS_90_donchian_high_50_z_sos: 227 NaNs (22.6%)
  LENS_90_logistic_prob_30_z_sos: 207 NaNs (20.6%)
  LENS_90_dispersion_30_z_sos: 207 NaNs (20.6%)
  LENS_90_adx_14_z_sos: 204 NaNs (20.3%)
  LENS_90_hma_21_pct_z_sos: 202 NaNs (20.1%)
  LENS_90_er_20_z_sos: 198 NaNs (19.7%)
  LENS_

Permutation scoring:   0%|          | 0/179 [00:00<?, ?it/s]

  ⚠️  ITER 20 REJECTED — val_acc 0.5162 < 0.52

✅ GLOBAL AUDIT COMPLETE
📊 DATA ROWS COLLECTED: 874
📂 CSV SAVED TO:        /content/drive/MyDrive/Sovereign_Titan_Results/Sovereign_Titan_v3.20.2_CoreVerified/Sovereign_Audit_Master_20260307_064427.csv

🛡️  DATA LEAKAGE CHECK (based on UNSEEN test data):
  DIRECTION    test_acc=0.5034  ⚠️  UNDERPERFORMING
  EASE         test_acc=0.5197  ⚠️  UNDERPERFORMING
  EXP          test_acc=0.5116  ⚠️  UNDERPERFORMING

═════════════════════════════════════════════════════════════════
🚀 FINAL SOVEREIGN ARRAYS (TOP 19 PER BRAIN)
═════════════════════════════════════════════════════════════════

💎 FINAL 19 — BRAIN: DIRECTION  (iters: 13/20  val=0.579  test=0.503)
RNK | FEATURE                                | PERSIST  | AVG_IMP 
──────────────────────────────────────────────────────────────
01  | WIN_10_psar_trend_pct                  | 11/13    | 0.9221
02  | WIN_10_psar_trend_slope_align_pct      |  9/13    | 0.8305
03  | WIN_10_di_slope_align_pct    

In [ ]:
# ==============================================================================
# SOVEREIGN TITAN v4.0 FINAL — COMPLETE WORKING SYSTEM
# Three Brains: DIRECTION (binary), EASE (tradability), EXPANSION (volatility)
# Features: Physics-First (log diff, ratios, PCA-19) + 10x speed + Full intelligence
# ==============================================================================
import os, gc, warnings, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from numba import jit
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import yfinance as yf
import random

# ==============================================================================
# ENVIRONMENT SETUP
# ==============================================================================
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_DIR = '/content/drive/MyDrive/Sovereign_Titan_Results/'
    EASE_PARQUET_PATH = '/content/drive/MyDrive/ease_data.parquet'
    ENV = 'COLAB'
except ImportError:
    BASE_OUTPUT_DIR = './results/'
    EASE_PARQUET_PATH = './ease_data.parquet'
    ENV = 'LOCAL'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    DEVICE = '/device:GPU:0'
    print(f"✅ GPU Active: {tf.test.gpu_device_name()}")
else:
    DEVICE = '/cpu:0'
    print("⚠️  CPU Mode (slower)")

TEST_NAME = "Sovereign_Titan_v4.0_Final"
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, TEST_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

print(f"[ENV] {ENV} | Workers: {N_WORKERS} | Device: {DEVICE}")
print(f"[OUTPUT] {OUTPUT_DIR}\n")

# ==============================================================================
# SYMBOLS
# ==============================================================================
TITAN_SYMBOLS = [
    'AA','AAL','AAPL','ABNB','ACWI','AEM','AFRM','AI','ALAB','ALB','AMAT','AMD','AMZN',
    'ANET','APA','APH','ARKK','AVGO','BA','BABA','BAC','BKR','BLDR','C','CARR','CAT',
    'CCJ','CCL','CE','CELH','CLF','CLSK','CMG','CNC','CPRT','CRM','CSCO','CSX','CVS',
    'CVX','DAL','DDOG','DHR','DIA','DIS','DKNG','DLTR','DOW','DVN','DXCM','EA','EBAY',
    'EEM','EMR','EQT','EWJ','EWT','EWW','EWY','EWZ','EXC','F','FANG','FCX','FITB',
    'FTNT','FTV','FXI','GBTC','GDX','GDXJ','GEHC','GFS','GIS','GOOG','GOOGL','GS',
    'HAL','HOOD','HPE','HPQ','HWM','IAU','IBM','IGV','IJH','IJR','INTC','IP','IR',
    'IWM','IYR','JNJ','KDP','KMI','KO','KRE','KWEB','LOW','LRCX','LUV','LVS','LYFT',
    'MAR','MARA','MCHP','MGM','MNST','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR',
    'MU','NCLH','NEE','NEM','NKE','NUE','NVDA','NVO','NXPI','ON','ORCL','OXY','PANW',
    'PCAR','PDD','PEP','PFE','PINS','PLTR','PYPL','QCOM','QQQ','QQQM','RBLX','RIOT',
    'RIVN','RTX','SBUX','SCHW','SHOP','SJM','SLB','SLV','SMCI','SMH','SNAP','SNOW',
    'SOFI','SOXX','SPLG','SPY','TER','TGT','TJX','TLT','TMUS','TQQQ','TSCO','TSLA',
    'TTD','TTWO','TWLO','TXN','U','UAL','UBER','UPS','USB','USO','VLO','VNQ','VRT',
    'VST','VT','VTR','WMT','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP',
    'XLRE','XLU','XLV','XLY','XOM','XOP','XRT'
]

# ==============================================================================
# BRAIN CONFIGURATION
# ==============================================================================
BRAIN_LOCKS = {'DIRECTION': [], 'EASE': [], 'EXP': []}
DATA_WINDOW = {'DIRECTION': '6y', 'EASE': '4y', 'EXP': '4y'}
MODEL_TYPE = {'DIRECTION': 'GRU', 'EASE': 'LSTM', 'EXP': 'LSTM'}
LOSS_TYPE = {'DIRECTION': 'binary_crossentropy', 'EASE': 'huber', 'EXP': 'huber'}
ACTIVATION = {'DIRECTION': 'sigmoid', 'EASE': 'linear', 'EXP': 'linear'}

# ==============================================================================
# FAST NUMBA KERNELS
# ==============================================================================
@jit(nopython=True, cache=True, fastmath=True)
def _fast_linslope(arr, window):
    n = len(arr); out = np.zeros(n); x_mean = (window - 1) / 2.0
    for i in range(window - 1, n):
        y_sum = 0.0
        for j in range(window): y_sum += arr[i - window + 1 + j]
        y_mean = y_sum / window
        num = 0.0; den = 0.0
        for j in range(window):
            dx = j - x_mean; dy = arr[i - window + 1 + j] - y_mean
            num += dx * dy; den += dx * dx
        out[i] = num / den if den != 0.0 else 0.0
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_zscore(arr, window):
    n = len(arr); out = np.zeros(n)
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window): s += arr[i - window + 1 + j]
        mean = s / window
        var = 0.0
        for j in range(window):
            d = arr[i - window + 1 + j] - mean
            var += d * d
        std = np.sqrt(var / window)
        out[i] = (arr[i] - mean) / (std + 1e-9)
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_shannon(arr, window, bins=10):
    n = len(arr); out = np.zeros(n)
    for i in range(window - 1, n):
        mn = arr[i - window + 1]; mx = arr[i - window + 1]
        for j in range(1, window):
            val = arr[i - window + 1 + j]
            if val < mn: mn = val
            if val > mx: mx = val
        if mx == mn: continue
        counts = np.zeros(bins)
        for j in range(window):
            val = arr[i - window + 1 + j]
            idx = int((val - mn) / (mx - mn) * bins)
            if idx >= bins: idx = bins - 1
            counts[idx] += 1.0
        entropy = 0.0
        for j in range(bins):
            p = counts[j] / window + 1e-9
            entropy -= p * np.log(p)
        out[i] = entropy
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_hurst(arr, window):
    n = len(arr); out = np.full(n, 0.5)
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window): s += arr[i - window + 1 + j]
        mean = s / window
        var = 0.0
        for j in range(window):
            d = arr[i - window + 1 + j] - mean
            var += d * d
        std = np.sqrt(var / window)
        if std < 1e-12: continue
        out[i] = np.log(std + 1e-9) / np.log(window)
        if np.isnan(out[i]): out[i] = 0.5
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_tema(price):
    n = len(price); ema1 = np.zeros(n); ema2 = np.zeros(n); ema3 = np.zeros(n)
    alpha = 2.0 / 31.0
    ema1[0] = price[0]; ema2[0] = price[0]; ema3[0] = price[0]
    for i in range(1, n):
        ema1[i] = alpha * price[i] + (1 - alpha) * ema1[i-1]
        ema2[i] = alpha * ema1[i] + (1 - alpha) * ema2[i-1]
        ema3[i] = alpha * ema2[i] + (1 - alpha) * ema3[i-1]
    return 3 * ema1 - 3 * ema2 + ema3

@jit(nopython=True, cache=True, fastmath=True)
def _fast_r_sq(arr, window):
    n = len(arr); out = np.zeros(n); x_mean = (window - 1) / 2.0
    for i in range(window - 1, n):
        y_sum = 0.0
        for j in range(window): y_sum += arr[i - window + 1 + j]
        y_mean = y_sum / window
        num = 0.0; den_x = 0.0; den_y = 0.0
        for j in range(window):
            dx = j - x_mean; dy = arr[i - window + 1 + j] - y_mean
            num += dx * dy; den_x += dx * dx; den_y += dy * dy
        if den_x == 0.0 or den_y == 0.0: continue
        r = num / (np.sqrt(den_x) * np.sqrt(den_y))
        out[i] = r * r
    return out

_d = np.random.randn(100).astype(np.float64)
_fast_linslope(_d, 10); _fast_zscore(_d, 10); _fast_shannon(_d, 20)
_fast_hurst(_d, 50); _fast_tema(_d); _fast_r_sq(_d, 30)
print("✅ Numba kernels compiled\n")

# ==============================================================================
# FEATURE FACTORY
# ==============================================================================
def generate_physics_features(df, brain_name='DIRECTION', ease_data=None):
    start = time.time()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.copy()
    df.columns = [c.lower() for c in df.columns]

    h = df['high'].values.astype(np.float64)
    l = df['low'].values.astype(np.float64)
    c = df['close'].values.astype(np.float64)
    o = df['open'].values.astype(np.float64) if 'open' in df.columns else c.copy()
    v = df['volume'].values.astype(np.float64)

    hlc = (h + l + c) / 3
    idx = df.index
    n = len(df)

    # Log differencing (stationarity)
    log_hlc = np.log(hlc + 1e-9)
    log_diff = np.zeros(n)
    log_diff[1:] = log_hlc[1:] - log_hlc[:-1]

    # Pure ratios (Sovereign Pillars)
    tema = _fast_tema(hlc)
    tema_ratio = hlc / (tema + 1e-9)

    sma_5 = pd.Series(hlc, index=idx).rolling(5).mean().values
    sma_5_ratio = hlc / (sma_5 + 1e-9)

    sma_20 = pd.Series(hlc, index=idx).rolling(20).mean().values
    sma_20_ratio = hlc / (sma_20 + 1e-9)

    hlc_s = pd.Series(hlc, index=idx)
    er_20 = (hlc_s.diff(20).abs() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values
    vidya_cmo = (hlc_s.diff().rolling(20).sum() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values

    # Predictability signatures
    hurst_50 = _fast_hurst(hlc, 50)
    hurst_20 = _fast_hurst(hlc, 20)
    shannon_20 = _fast_shannon(hlc, 20)
    shannon_10 = _fast_shannon(hlc, 10)

    # Trend quality
    r_sq_30 = _fast_r_sq(hlc, 30)
    r_sq_10 = _fast_r_sq(hlc, 10)
    linreg_slope_30 = _fast_linslope(hlc, 30)

    # ADX
    tr = np.maximum(h - l, np.maximum(np.abs(h - np.roll(c, 1)), np.abs(l - np.roll(c, 1))))
    tr[0] = h[0] - l[0]
    atr_14 = pd.Series(tr, index=idx).rolling(14).mean().values

    dm_plus = np.maximum(h - np.roll(h, 1), 0)
    dm_minus = np.maximum(np.roll(l, 1) - l, 0)
    dm_plus[0] = 0; dm_minus[0] = 0

    pdi = 100 * pd.Series(dm_plus, index=idx).rolling(14).mean().values / (atr_14 + 1e-9)
    mdi = 100 * pd.Series(dm_minus, index=idx).rolling(14).mean().values / (atr_14 + 1e-9)
    adx = (100 * np.abs(pdi - mdi) / (pdi + mdi + 1e-9))
    adx = pd.Series(adx, index=idx).rolling(14).mean().values

    # Top composites
    shannon_ratio_10_20 = shannon_10 / (shannon_20 + 1e-9)
    entropy_r_sq_ratio = shannon_20 / (r_sq_30 + 1e-9)
    r_sq_ratio_10_30 = r_sq_10 / (r_sq_30 + 1e-9)
    er_10 = (hlc_s.diff(10).abs() / (hlc_s.diff().abs().rolling(10).sum() + 1e-9)).values
    er_ratio_10_20 = er_10 / (er_20 + 1e-9)

    sma_30 = pd.Series(hlc, index=idx).rolling(30).mean().values
    d_sma = hlc / (sma_30 + 1e-9)
    d_tema = hlc / (tema + 1e-9)
    dispersion = np.std(np.stack([d_sma, d_tema], axis=1), axis=1)
    dispersion_r_sq = dispersion / (r_sq_30 + 1e-9)

    # 19 Physics Seeds
    PHYSICS_SEEDS = {
        'log_diff': log_diff, 'tema_ratio': tema_ratio, 'sma_5_ratio': sma_5_ratio,
        'sma_20_ratio': sma_20_ratio, 'er_20': er_20, 'vidya_cmo': vidya_cmo,
        'hurst_50': hurst_50, 'hurst_20': hurst_20, 'shannon_20': shannon_20,
        'shannon_10': shannon_10, 'r_sq_30': r_sq_30, 'r_sq_10': r_sq_10,
        'linreg_slope_30': linreg_slope_30, 'adx_14': adx,
        'shannon_ratio_10_20': shannon_ratio_10_20, 'entropy_r_sq_ratio': entropy_r_sq_ratio,
        'r_sq_ratio_10_30': r_sq_ratio_10_30, 'er_ratio_10_20': er_ratio_10_20,
        'dispersion_r_sq': dispersion_r_sq,
    }

    # Triple-order lenses (60-day window - optimal regime-invariant horizon)
    for name, arr in PHYSICS_SEEDS.items():
        z = _fast_zscore(arr, 60)
        z_slope = _fast_linslope(z, 60)
        z_sos = _fast_linslope(z_slope, 60)
        df[f'{name}_z'] = z
        df[f'{name}_z_slope'] = z_slope
        df[f'{name}_z_sos'] = z_sos

    # TARGETS (YOUR ORIGINAL)
    if brain_name == 'DIRECTION':
        df['T_FINAL'] = (pd.Series(c, index=idx).shift(-1) > c).astype('float')
        df.loc[df.index[-1], 'T_FINAL'] = np.nan

    elif brain_name == 'EASE':
        if ease_data is not None:
            try:
                ease_data = ease_data.copy()
                ease_data['date'] = pd.to_datetime(ease_data['date'])
                df_reset = df.reset_index()
                df_reset['date'] = pd.to_datetime(df_reset.index)

                if 'symbol' in df_reset.columns:
                    merged = df_reset.merge(ease_data[['date', 'symbol', 'EASE_val']],
                                           on=['date', 'symbol'], how='left')
                    df['EASE_val'] = merged['EASE_val'].values
                    df['T_FINAL'] = pd.Series(df['EASE_val'].values, index=idx).shift(-1)
                else:
                    df['T_FINAL'] = np.nan
            except Exception as e:
                print(f"  ⚠️  EASE merge failed: {e}")
                df['T_FINAL'] = np.nan
        else:
            print("  ⚠️  No EASE data - using proxy")
            price_move = (c - o) / (atr_14 + 1e-9)
            df['T_FINAL'] = pd.Series(price_move, index=idx).shift(-1)

    elif brain_name == 'EXP':
        daily_range = h - l
        avg_range_20 = pd.Series(daily_range, index=idx).rolling(20).mean()
        range_tomorrow = pd.Series(daily_range, index=idx).shift(-1)
        df['T_FINAL'] = range_tomorrow / (avg_range_20 + 1e-9)
        df['T_FINAL'].iloc[-1] = np.nan

    # Cleanup
    df = df.replace([np.inf, -np.inf], np.nan)
    feature_cols = [c for c in df.columns if c.endswith(('_z', '_slope', '_sos'))]

    initial_len = len(df)

    # Count NaNs per column
    nan_counts = df[feature_cols + ['T_FINAL']].isna().sum()
    max_nans = nan_counts.max()

    df = df.dropna(subset=feature_cols + ['T_FINAL'])
    dropped = initial_len - len(df)

    elapsed = time.time() - start

    if len(df) == 0:
        print(f"  ⚠️  ALL ROWS DROPPED! Max NaNs in any column: {max_nans}/{initial_len}")
    else:
        print(f"  [PHYSICS] {len(feature_cols)} features in {elapsed:.1f}s | {brain_name} target")
        print(f"  [CLEANUP] {dropped}/{initial_len} dropped, {len(df)} remaining")

    return df

# ==============================================================================
# PARALLEL DATA LOADING
# ==============================================================================
def _fetch_symbol(args):
    sym, period = args
    try:
        df = yf.download(sym, period=period, progress=False, auto_adjust=True, threads=False)
        if df.empty or len(df) < 300: return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.columns = [c.lower() for c in df.columns]
        if not all(c in df.columns for c in ['high', 'low', 'close', 'volume']):
            return None
        return (sym, df)
    except Exception:
        return None

def _process_features(args):
    sym, df, brain_name, ease_data = args
    try:
        result = generate_physics_features(df, brain_name, ease_data)
        if not result.empty:
            result['symbol'] = sym
            print(f"  ✅ {sym}: {len(result)} rows")
        else:
            print(f"  ❌ {sym}: empty after features")
        return result
    except Exception as e:
        print(f"  ❌ {sym}: {str(e)[:50]}")
        return pd.DataFrame()

def load_data_parallel(symbols, period, brain_name, ease_data=None):
    print(f"📥 Downloading {len(symbols)} symbols...")

    with ThreadPoolExecutor(max_workers=min(20, len(symbols))) as executor:
        results = list(tqdm(
            executor.map(_fetch_symbol, [(s, period) for s in symbols]),
            total=len(symbols), desc="Download"
        ))

    valid = [r for r in results if r is not None]
    print(f"   ✅ {len(valid)}/{len(symbols)} fetched\n")
    if not valid: return pd.DataFrame()

    print(f"⚙️  Features (workers={N_WORKERS})...")

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        processed = list(tqdm(
            executor.map(_process_features, [(s, d, brain_name, ease_data) for s, d in valid]),
            total=len(valid), desc="Features"
        ))

    processed = [df for df in processed if not df.empty]
    if not processed: return pd.DataFrame()

    master = pd.concat(processed, axis=0, ignore_index=True)
    print(f"   ✅ Master: {len(master):,} rows\n")
    return master

# ==============================================================================
# MODEL
# ==============================================================================
def build_brain_model(n_features, brain_name, seq_len=60):
    model_type = MODEL_TYPE[brain_name]
    activation = ACTIVATION[brain_name]
    loss_type = LOSS_TYPE[brain_name]

    with tf.device(DEVICE):
        model = Sequential([
            Input(shape=(seq_len, n_features)),
            GRU(128, return_sequences=True) if model_type == 'GRU' else LSTM(128, return_sequences=True),
            BatchNormalization(),
            Dropout(0.2),
            GRU(64) if model_type == 'GRU' else LSTM(64),
            BatchNormalization(),
            Dropout(0.2),
            Dense(32, activation='relu'),
            Dropout(0.1),
            Dense(1, activation=activation, dtype='float32')
        ])

        loss = tf.keras.losses.Huber(delta=1.0) if loss_type == 'huber' else loss_type
        metrics = ['accuracy'] if brain_name == 'DIRECTION' else ['mae']

        model.compile(optimizer=Adam(1e-3), loss=loss, metrics=metrics)

    return model

# ==============================================================================
# TRAINING & AUDIT
# ==============================================================================
def run_physics_audit(master_df, brain_name, seq_len=60, epochs=50):
    feature_cols = [c for c in master_df.columns if c.endswith(('_z', '_slope', '_sos'))]

    X_raw = master_df[feature_cols].values.astype(np.float32)
    y_raw = master_df['T_FINAL'].values.astype(np.float32)

    n = len(X_raw)
    X_seqs = np.stack([X_raw[i-seq_len:i] for i in range(seq_len, n)])
    y_seqs = y_raw[seq_len:]

    train_end = int(len(X_seqs) * 0.70)
    val_end = int(len(X_seqs) * 0.85)

    X_tr = X_seqs[:train_end]
    y_tr = y_seqs[:train_end]
    X_val = X_seqs[train_end:val_end]
    y_val = y_seqs[train_end:val_end]
    X_test = X_seqs[val_end:]
    y_test = y_seqs[val_end:]

    print(f"  [SPLITS] Train={len(X_tr):,} | Val={len(X_val):,} | Test={len(X_test):,}")

    # Scale (TRAIN ONLY - skip warmup contaminated rows)
    scaler = RobustScaler()
    n_train, seq, feats = X_tr.shape
    X_tr_2d = X_tr.reshape(-1, feats)

    # Skip first 350 rows (60-day features × 3 transforms + buffer)
    WARMUP_SKIP = 350
    if len(X_tr_2d) > WARMUP_SKIP:
        scaler.fit(X_tr_2d[WARMUP_SKIP:])
        print(f"  [SCALING] Fitted on rows {WARMUP_SKIP}-{len(X_tr_2d)} (excluded warmup)")
    else:
        scaler.fit(X_tr_2d)
        print(f"  [SCALING] Fitted on all {len(X_tr_2d)} rows (insufficient for warmup exclusion)")

    X_tr_scaled = scaler.transform(X_tr_2d).reshape(n_train, seq, feats)
    X_val_scaled = scaler.transform(X_val.reshape(-1, feats)).reshape(len(X_val), seq, feats)
    X_test_scaled = scaler.transform(X_test.reshape(-1, feats)).reshape(len(X_test), seq, feats)

    # PCA to 19
    print(f"  [PCA] {feats} → 19 components...")
    pca = PCA(n_components=19)
    X_tr_flat = X_tr_scaled.reshape(-1, feats)
    pca.fit(X_tr_flat[WARMUP_SKIP:] if len(X_tr_flat) > WARMUP_SKIP else X_tr_flat)

    var_retained = np.sum(pca.explained_variance_ratio_)
    print(f"  [PCA] Variance: {var_retained*100:.2f}%")

    X_tr_pca = pca.transform(X_tr_scaled.reshape(-1, feats)).reshape(n_train, seq, 19)
    X_val_pca = pca.transform(X_val_scaled.reshape(-1, feats)).reshape(len(X_val), seq, 19)
    X_test_pca = pca.transform(X_test_scaled.reshape(-1, feats)).reshape(len(X_test), seq, 19)

    # Train
    model = build_brain_model(19, brain_name, seq_len)

    print(f"\n  [TRAIN] {MODEL_TYPE[brain_name]} | {LOSS_TYPE[brain_name]} | {epochs} epochs")

    train_ds = (tf.data.Dataset.from_tensor_slices((X_tr_pca, y_tr))
                .shuffle(20000).batch(512).prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val_pca, y_val))
              .batch(1024).prefetch(tf.data.AUTOTUNE))

    with tf.device(DEVICE):
        model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                  callbacks=[
                      EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                      ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
                  ],
                  verbose=1)

    # Evaluate
    is_classification = (brain_name == 'DIRECTION')

    if is_classification:
        val_loss, val_metric = model.evaluate(X_val_pca, y_val, verbose=0)
        test_loss, test_metric = model.evaluate(X_test_pca, y_test, verbose=0)
        metric_name = "Acc"
    else:
        val_loss, val_metric = model.evaluate(X_val_pca, y_val, verbose=0)
        test_loss, test_metric = model.evaluate(X_test_pca, y_test, verbose=0)
        metric_name = "MAE"

    print(f"\n  [RESULTS] {metric_name}")
    print(f"    Val:  {val_metric:.4f}")
    print(f"    Test: {test_metric:.4f}")

    if is_classification:
        if test_metric > 0.70:
            print(f"    🚨 LEAKAGE")
        elif val_metric > 0.70 and test_metric < 0.65:
            print(f"    ✅ OVERFIT (acceptable)")
        else:
            print(f"    ✅ NO LEAKAGE")

    # Importance (PCA loadings)
    print(f"\n  [IMPORTANCE] Computing...")
    components = pca.components_

    report_rows = []
    for fi, feat_name in enumerate(tqdm(feature_cols, desc="Scoring")):
        contrib = np.abs(components[:, fi]).sum()
        report_rows.append({'Feature': feat_name, 'I_raw': contrib})

    del model
    gc.collect()
    tf.keras.backend.clear_session()

    return pd.DataFrame(report_rows), val_metric, test_metric, var_retained

# ==============================================================================
# SOVEREIGN SELECTION
# ==============================================================================
def apply_sovereign_hunt(ledger_df, master_data_df, brain_name, max_slots=19):
    locked_list = BRAIN_LOCKS.get(brain_name, [])
    candidates = ledger_df.sort_values(by='I_raw', ascending=False)
    picked = [f for f in locked_list if f in ledger_df['Feature'].values]

    CORR_THRESHOLD = 0.85

    feat_cols = [c for c in master_data_df.columns if c.endswith(('_z', '_slope', '_sos'))]
    corr_matrix = master_data_df[feat_cols].corr()

    for _, row in candidates.iterrows():
        if len(picked) >= max_slots: break
        f_name = row['Feature']
        if f_name in picked: continue
        if len(picked) > 0 and corr_matrix[f_name].loc[picked].max() > CORR_THRESHOLD:
            continue
        picked.append(f_name)

    pca = PCA()
    pca.fit(RobustScaler().fit_transform(master_data_df[picked]))
    var_map = np.cumsum(pca.explained_variance_ratio_)

    return picked, var_map

def generate_judicial_ledger(brain_name, report_df, master_data_df, iteration=1):
    df = report_df.copy()
    df['I_Norm'] = (df['I_raw'] - df['I_raw'].min()) / (df['I_raw'].max() - df['I_raw'].min() + 1e-9)

    active_picks, var_map = apply_sovereign_hunt(df, master_data_df, brain_name)
    corr_sub = master_data_df[active_picks].corr().abs()
    avg_corr = (corr_sub.sum().sum() - len(active_picks)) / (len(active_picks)**2 - len(active_picks) + 1e-9)

    print(f"\n╔══ {brain_name} SOVEREIGN CORE v4.0 (Iter {iteration}) ══╗")
    print(f"║ {'RNK':<3} | {'FEATURE':<35} | {'UV%':<4} | {'mR':<4} | {'IMPACT':<8} ║")
    print("╠" + "═"*4 + "╬" + "═"*37 + "╬" + "═"*6 + "╬" + "═"*6 + "╬" + "═"*10 + "╣")

    for i, f_name in enumerate(active_picks):
        f_row = df[df['Feature'] == f_name].iloc[0]
        is_locked = f_name in BRAIN_LOCKS.get(brain_name, [])
        icon = "🔒" if is_locked else "🔭"

        other_picks = [p for p in active_picks if p != f_name]
        max_r = corr_sub[f_name].loc[other_picks].max() if other_picks else 0.0
        uv_val = (1 - corr_sub[f_name].loc[other_picks].mean()) * 100 if other_picks else 100.0

        print(f"║ {i+1:02d}  | {icon} {f_name[:33]:<33} | {uv_val:>3.0f}% | {max_r:.2f} | {f_row['I_Norm']:.4f} ║")

        df.loc[df['Feature'] == f_name, ['UV%','Max_R','Is_Locked']] = [uv_val, max_r, is_locked]

    total_var = var_map[-1] if len(var_map) > 0 else 0
    print("╠" + "═"*73 + "╣")
    print(f"║ PCA VARIANCE: {total_var*100:>48.2f}% ║")
    print(f"║ AVG CORRELATION: {avg_corr:>45.3f} ║")
    print(f"║ SLOTS: {len(active_picks):>55}/19 ║")
    print("╚" + "═"*73 + "╝")

    df['UV%'] = df['UV%'].fillna(0.0)
    df['Max_R'] = df['Max_R'].fillna(0.0)
    df['Is_Locked'] = df['Is_Locked'].fillna(False)
    df['In_Top_19'] = df['Feature'].isin(active_picks)

    return df

# ==============================================================================
# MAIN
# ==============================================================================
if __name__ == '__main__':
    print("\n" + "="*70)
    print("   SOVEREIGN TITAN v4.0 FINAL — COMPLETE SYSTEM")
    print("="*70)
    print("  🧠 DIRECTION: Tomorrow UP? (binary)")
    print("  🧠 EASE: Tomorrow's 3min tradability (regression)")
    print("  🧠 EXP: Tomorrow's range expansion (regression)")
    print("  🔬 Physics-First: Log diff, ratios, PCA-19, 60-day lenses")
    print("  ⚡ 10x Speed: Numba + parallel")
    print("="*70 + "\n")

    choice = input("Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
    BRAINS = ['DIRECTION','EASE','EXP'] if choice == '4' else [
        {'1':'DIRECTION','2':'EASE','3':'EXP'}[choice]
    ]

    num_symbols = int(input("Symbols/iteration [50]: ") or "50")
    num_iters = int(input("Iterations [20]: ") or "20")

    # Load EASE data if needed
    ease_data = None
    if 'EASE' in BRAINS:
        try:
            ease_data = pd.read_parquet(EASE_PARQUET_PATH)
            print(f"✅ EASE data: {len(ease_data)} rows\n")
        except Exception:
            print(f"⚠️  No EASE data at {EASE_PARQUET_PATH}\n")

    final_report = []

    for BRAIN in BRAINS:
        print(f"\n[BRAIN] {BRAIN} | {MODEL_TYPE[BRAIN]} | {DATA_WINDOW[BRAIN]}")

        for it in range(1, num_iters + 1):
            print(f"\n{'═'*70}")
            print(f"  ITERATION {it}/{num_iters}  —  {BRAIN}")
            print(f"{'═'*70}\n")

            pool = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
            master_df = load_data_parallel(pool, DATA_WINDOW[BRAIN], BRAIN, ease_data)

            if master_df.empty:
                print("  ⚠️  No data")
                continue

            report_raw, val_metric, test_metric, pca_var = run_physics_audit(master_df, BRAIN)

            # Quality gate
            if BRAIN == 'DIRECTION' and val_metric < 0.50:
                print(f"  ⚠️  Val {val_metric:.4f} < 0.50 - skipping")
                gc.collect()
                tf.keras.backend.clear_session()
                continue

            iteration_ledger = generate_judicial_ledger(BRAIN, report_raw, master_df, it)
            iteration_ledger['Iteration'] = it
            iteration_ledger['Brain'] = BRAIN
            iteration_ledger['Model_Type'] = MODEL_TYPE[BRAIN]
            iteration_ledger['Data_Window'] = DATA_WINDOW[BRAIN]
            iteration_ledger['Val_Metric'] = val_metric
            iteration_ledger['Test_Metric'] = test_metric
            iteration_ledger['PCA_Var'] = pca_var
            iteration_ledger['Timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            final_report.append(iteration_ledger)

            gc.collect()
            tf.keras.backend.clear_session()

    # Export
    if final_report:
        raw_df = pd.concat(final_report, axis=0)

        stats = (raw_df.groupby(['Brain','Feature'])
                 .agg(Persistence=('Feature','count'),
                      A_Impact=('I_Norm','mean'),
                      A_UV=('UV%','mean'))
                 .reset_index())

        final_df = (raw_df.merge(stats, on=['Brain','Feature'], how='left')
                          .sort_values(['Brain','Persistence','A_Impact'], ascending=False))

        csv_path = os.path.join(OUTPUT_DIR, f"Sovereign_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        final_df.to_csv(csv_path, index=False)

        print("\n" + "="*70)
        print("✅ COMPLETE")
        print(f"📊 Rows: {len(raw_df)}")
        print(f"📂 {csv_path}")

        print("\n" + "="*70)
        print("📊 RESULTS BY BRAIN:")
        for brain in final_df['Brain'].unique():
            brain_data = raw_df[raw_df['Brain'] == brain]
            avg_metric = brain_data['Test_Metric'].mean()

            if brain == 'DIRECTION':
                if avg_metric > 0.70: status = "🚨 LEAKAGE"
                elif avg_metric > 0.50: status = "✅ SIGNAL"
                else: status = "⚠️  NOISE"
                print(f"  {brain:<12} Test Acc: {avg_metric:.4f}  {status}")
            else:
                print(f"  {brain:<12} Test MAE: {avg_metric:.4f}")
        print("="*70)

In [ ]:
# ==============================================================================
# SOVEREIGN TITAN v4.0 — SCOTTIE PIPPEN EDITION
# CPU-Optimized | All 37 Trend-Only Indicators | Restored Ratios | 233 Features → PCA to 19
# Includes: HMA, FWMA, Velocity/Acceleration, EMA 50/200, Trend Factors, Mass Index
# ==============================================================================
import os, gc, warnings, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import tensorflow as tf

# ==============================================================================
# ENVIRONMENT DETECTION
# ==============================================================================
def detect_environment():
    try:
        import google.colab
        return 'COLAB'
    except ImportError:
        return 'LOCAL'

ENV = detect_environment()

if ENV == 'COLAB':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_DIR = '/content/drive/MyDrive/Sovereign_Titan_Results/'
    EASE_PARQUET_PATH =  '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
else:
    BASE_OUTPUT_DIR = './results/'
    EASE_PARQUET_PATH = './ease_data.parquet'

TEST_NAME = "Sovereign_Titan_v4_Scottie_Pippen_Edition"
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, TEST_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[ENV] {ENV}")
print(f"[OUTPUT] {OUTPUT_DIR}\n")

# ==============================================================================
# GPU/CPU SETUP
# ==============================================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU detected: {len(gpus)} device(s)")
    DEVICE = '/device:GPU:0'
else:
    print("⚠️  No GPU - using CPU (slower but works)")
    DEVICE = '/cpu:0'

print(f"[DEVICE] {DEVICE}\n")

# ==============================================================================
# IMPORTS
# ==============================================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from numba import jit
from datetime import datetime
import yfinance as yf
import random

N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# ==============================================================================
# SYMBOLS
# ==============================================================================
TITAN_SYMBOLS = [
    'AA','AAL','AAPL','ABNB','ACWI','AEM','AFRM','AI','ALAB','ALB','AMAT','AMD','AMZN',
    'ANET','APA','APH','ARKK','AVGO','BA','BABA','BAC','BKR','BLDR','C','CARR','CAT',
    'CCJ','CCL','CE','CELH','CLF','CLSK','CMG','CNC','CPRT','CRM','CSCO','CSX','CVS',
    'CVX','DAL','DDOG','DHR','DIA','DIS','DKNG','DLTR','DOW','DVN','DXCM','EA','EBAY',
    'EEM','EMR','EQT','EWJ','EWT','EWW','EWY','EWZ','EXC','F','FANG','FCX','FITB',
    'FTNT','FTV','FXI','GBTC','GDX','GDXJ','GEHC','GFS','GIS','GOOG','GOOGL','GS',
    'HAL','HOOD','HPE','HPQ','HWM','IAU','IBM','IGV','IJH','IJR','INTC','IP','IR',
    'IWM','IYR','JNJ','KDP','KMI','KO','KRE','KWEB','LOW','LRCX','LUV','LVS','LYFT',
    'MAR','MARA','MCHP','MGM','MNST','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR',
    'MU','NCLH','NEE','NEM','NKE','NUE','NVDA','NVO','NXPI','ON','ORCL','OXY','PANW',
    'PCAR','PDD','PEP','PFE','PINS','PLTR','PYPL','QCOM','QQQ','QQQM','RBLX','RIOT',
    'RIVN','RTX','SBUX','SCHW','SHOP','SJM','SLB','SLV','SMCI','SMH','SNAP','SNOW',
    'SOFI','SOXX','SPLG','SPY','TER','TGT','TJX','TLT','TMUS','TQQQ','TSCO','TSLA',
    'TTD','TTWO','TWLO','TXN','U','UAL','UBER','UPS','USB','USO','VLO','VNQ','VRT',
    'VST','VT','VTR','WMT','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP',
    'XLRE','XLU','XLV','XLY','XOM','XOP','XRT'
]

# ==============================================================================
# BRAIN CONFIGURATION
# ==============================================================================

# FAST MODE - Set to True for 3-5x faster training with minimal accuracy loss
FAST_MODE = True  # Change to False for full accuracy mode

# IMPORTANCE-ONLY MODE - Skip training, just get feature rankings (ultra-fast!)
# ⚠️  CRITICAL: This shows VARIANCE, not ALPHA!
# ⚠️  PCA importance ≠ Predictive utility
# ⚠️  Use ONLY for initial noise pruning, NOT for final feature selection
IMPORTANCE_ONLY = False  # Set True ONLY for rapid noise screening (~30 sec vs 3 min)

if FAST_MODE:
    print("⚡ FAST MODE ENABLED - Training will be 3-5x faster")
    EPOCHS = 6          # vs 50 in normal mode
    BATCH_SIZE = 2048    # vs 512 in normal mode
    SEQ_LEN = 30         # vs 60 in normal mode
    EARLY_STOP_PATIENCE = 4   # vs 10
    LR_REDUCE_PATIENCE = 2    # vs 5
else:
    EPOCHS = 25
    BATCH_SIZE = 512
    SEQ_LEN = 60
    EARLY_STOP_PATIENCE = 10
    LR_REDUCE_PATIENCE = 5

BRAIN_LOCKS = {'DIRECTION': ['hma_21_ratio_z','shannon_ratio_10_20_z',' fwma_13_ratio_z','entropy_r_sq_ratio_z'], 'EASE': [], 'EXP': []}
DATA_WINDOW = {'DIRECTION': '4y', 'EASE': '4y', 'EXP': '4y'}
MODEL_TYPE = {'DIRECTION': 'GRU', 'EASE': 'LSTM', 'EXP': 'LSTM'}
LOSS_TYPE = {'DIRECTION': 'binary_crossentropy', 'EASE': 'huber', 'EXP': 'huber'}
ACTIVATION = {'DIRECTION': 'sigmoid', 'EASE': 'linear', 'EXP': 'linear'}

# ==============================================================================
# NUMBA KERNELS (FAST MATH OPERATIONS)
# ==============================================================================
@jit(nopython=True, cache=True, fastmath=True)
def _fast_linslope(arr, window):
    n = len(arr); out = np.zeros(n); x_mean = (window - 1) / 2.0
    for i in range(window - 1, n):
        y_sum = 0.0
        for j in range(window): y_sum += arr[i - window + 1 + j]
        y_mean = y_sum / window
        num = 0.0; den = 0.0
        for j in range(window):
            dx = j - x_mean; dy = arr[i - window + 1 + j] - y_mean
            num += dx * dy; den += dx * dx
        out[i] = num / den if den != 0.0 else 0.0
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_zscore(arr, window):
    n = len(arr); out = np.zeros(n)
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window): s += arr[i - window + 1 + j]
        mean = s / window
        var = 0.0
        for j in range(window):
            d = arr[i - window + 1 + j] - mean
            var += d * d
        std = np.sqrt(var / window)
        out[i] = (arr[i] - mean) / (std + 1e-9)
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_shannon(arr, window, bins=10):
    n = len(arr); out = np.zeros(n)
    for i in range(window - 1, n):
        mn = arr[i - window + 1]; mx = arr[i - window + 1]
        for j in range(1, window):
            val = arr[i - window + 1 + j]
            if val < mn: mn = val
            if val > mx: mx = val
        if mx == mn: continue
        counts = np.zeros(bins)
        for j in range(window):
            val = arr[i - window + 1 + j]
            idx = int((val - mn) / (mx - mn) * bins)
            if idx >= bins: idx = bins - 1
            counts[idx] += 1.0
        entropy = 0.0
        for j in range(bins):
            p = counts[j] / window + 1e-9
            entropy -= p * np.log(p)
        out[i] = entropy
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_hurst(arr, window):
    n = len(arr); out = np.full(n, 0.5)
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window): s += arr[i - window + 1 + j]
        mean = s / window
        var = 0.0
        for j in range(window):
            d = arr[i - window + 1 + j] - mean
            var += d * d
        std = np.sqrt(var / window)
        if std < 1e-12: continue
        out[i] = np.log(std + 1e-9) / np.log(window)
        if np.isnan(out[i]): out[i] = 0.5
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_tema(price):
    n = len(price); ema1 = np.zeros(n); ema2 = np.zeros(n); ema3 = np.zeros(n)
    alpha = 2.0 / 31.0
    ema1[0] = price[0]; ema2[0] = price[0]; ema3[0] = price[0]
    for i in range(1, n):
        ema1[i] = alpha * price[i] + (1 - alpha) * ema1[i-1]
        ema2[i] = alpha * ema1[i] + (1 - alpha) * ema2[i-1]
        ema3[i] = alpha * ema2[i] + (1 - alpha) * ema3[i-1]
    return 3 * ema1 - 3 * ema2 + ema3

@jit(nopython=True, cache=True, fastmath=True)
def _fast_r_sq(arr, window):
    n = len(arr); out = np.zeros(n); x_mean = (window - 1) / 2.0
    for i in range(window - 1, n):
        y_sum = 0.0
        for j in range(window): y_sum += arr[i - window + 1 + j]
        y_mean = y_sum / window
        num = 0.0; den_x = 0.0; den_y = 0.0
        for j in range(window):
            dx = j - x_mean; dy = arr[i - window + 1 + j] - y_mean
            num += dx * dy; den_x += dx * dx; den_y += dy * dy
        if den_x == 0.0 or den_y == 0.0: continue
        r = num / (np.sqrt(den_x) * np.sqrt(den_y))
        out[i] = r * r
    return out

# ==============================================================================
# RESEARCH-VALIDATED ADDITIONAL KERNELS
# ==============================================================================

@jit(nopython=True, cache=True, fastmath=True)
def _fast_wma(arr, window):
    """Weighted Moving Average"""
    n = len(arr)
    out = np.zeros(n)
    w_sum = window * (window + 1) / 2.0
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += arr[i - window + 1 + j] * (j + 1)
        out[i] = s / w_sum
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_fwma(arr, fib_weights):
    """Fibonacci Weighted Moving Average"""
    n = len(arr)
    window = len(fib_weights)
    out = np.zeros(n)
    w_sum = np.sum(fib_weights)

    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += arr[i - window + 1 + j] * fib_weights[j]
        out[i] = s / w_sum
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _fast_polynomial_fit(arr, window):
    """Fit second-order polynomial: A0 + A1*t + A2*t^2
    Returns velocity (A1) and acceleration (A2) arrays"""
    n = len(arr)
    velocity = np.zeros(n)
    acceleration = np.zeros(n)

    for i in range(window - 1, n):
        y = arr[i - window + 1 : i + 1]
        t = np.arange(window, dtype=np.float64)

        t_mean = np.mean(t)
        y_mean = np.mean(y)

        t_centered = t - t_mean
        y_centered = y - y_mean

        t2 = t_centered * t_centered
        t3 = t2 * t_centered
        t4 = t2 * t2

        sum_t2 = np.sum(t2)
        sum_t3 = np.sum(t3)
        sum_t4 = np.sum(t4)
        sum_ty = np.sum(t_centered * y_centered)
        sum_t2y = np.sum(t2 * y_centered)

        denom = sum_t2 * sum_t4 - sum_t3 * sum_t3
        if abs(denom) > 1e-10:
            velocity[i] = (sum_t4 * sum_ty - sum_t3 * sum_t2y) / denom
            acceleration[i] = (sum_t2 * sum_t2y - sum_t3 * sum_ty) / denom

    return velocity, acceleration

@jit(nopython=True, cache=True, fastmath=True)
def _fast_mass_index(high, low, window=25):
    """Mass Index: Sum of (EMA9(High-Low) / EMA9(EMA9(High-Low)))"""
    n = len(high)
    out = np.zeros(n)

    range_arr = high - low
    alpha = 2.0 / 10.0  # 9 + 1

    ema1 = np.zeros(n)
    ema1[0] = range_arr[0]
    for i in range(1, n):
        ema1[i] = alpha * range_arr[i] + (1 - alpha) * ema1[i-1]

    ema2 = np.zeros(n)
    ema2[0] = ema1[0]
    for i in range(1, n):
        ema2[i] = alpha * ema1[i] + (1 - alpha) * ema2[i-1]

    ratio = ema1 / (ema2 + 1e-9)

    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += ratio[i - window + 1 + j]
        out[i] = s

    return out

@jit(nopython=True, cache=True, fastmath=True)
def _kalman_numba(price, r=0.0001, q=0.001):
    """Kalman filter for adaptive smoothing"""
    x_hat = np.zeros_like(price)
    p = np.zeros_like(price)
    x_hat[0] = price[0]
    p[0] = 1.0
    for t in range(1, len(price)):
        p_minus = p[t-1] + q
        k = p_minus / (p_minus + r)
        x_hat[t] = x_hat[t-1] + k * (price[t] - x_hat[t-1])
        p[t] = (1 - k) * p_minus
    return x_hat

@jit(nopython=True, cache=True, fastmath=True)
def _rolling_cog(arr, window):
    """Center of Gravity (Ehler's)"""
    n = len(arr)
    out = np.zeros(n)
    for i in range(window - 1, n):
        num = 0.0
        den = 0.0
        for j in range(window):
            w = float(j + 1)
            num += w * arr[i - window + 1 + j]
            den += arr[i - window + 1 + j]
        out[i] = -num / (den + 1e-9)
    return out

@jit(nopython=True, cache=True, fastmath=True)
def _vhf_nb(price, window=28):
    """Vertical Horizontal Filter"""
    n = len(price)
    vhf = np.zeros(n)
    if window >= n:
        return vhf
    for i in range(window, n):
        segment = price[i-window+1:i+1]
        hcp = np.max(segment) - np.min(segment)
        sum_changes = 0.0
        for j in range(1, window):
            sum_changes += abs(segment[j] - segment[j-1])
        vhf[i] = hcp / (sum_changes + 1e-9)
    return vhf

@jit(nopython=True, cache=True, fastmath=True)
def _psar_nb(high, low, close, af_start=0.02, af_max=0.2):
    """Parabolic SAR"""
    n = len(close)
    psar = np.zeros(n)
    trend = np.ones(n)
    if n < 2:
        return psar, trend
    psar[0] = low[0]
    af = af_start
    ep = high[0]
    for i in range(1, n):
        psar[i] = psar[i-1] + af * (ep - psar[i-1])
        if trend[i-1] == 1:
            psar[i] = min(psar[i], low[i-1], low[i-2] if i > 1 else low[i-1])
            if low[i] < psar[i]:
                trend[i] = -1
                psar[i] = ep
                ep = low[i]
                af = af_start
            else:
                trend[i] = 1
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_start, af_max)
        else:
            psar[i] = max(psar[i], high[i-1], high[i-2] if i > 1 else high[i-1])
            if high[i] > psar[i]:
                trend[i] = 1
                psar[i] = ep
                ep = high[i]
                af = af_start
            else:
                trend[i] = -1
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_start, af_max)
    return psar, trend


@jit(nopython=True, cache=True)
def _choppiness_nb(high, low, close, period=14):
    n = len(close)
    chop = np.full(n, 50.0)
    if period >= n: return chop
    for i in range(period, n):
        atr_sum = 0.0
        for j in range(i-period+1, i+1):
            tr = max(high[j]-low[j],
                    abs(high[j]-close[j-1]) if j>0 else 0,
                    abs(low[j]-close[j-1]) if j>0 else 0)
            atr_sum += tr
        max_high = np.max(high[i-period+1:i+1])
        min_low = np.min(low[i-period+1:i+1])
        chop[i] = 100 * np.log10(atr_sum / (max_high - min_low + 1e-9)) / np.log10(period)
    return chop

@jit(nopython=True, cache=True)
def _fractal_energy_nb(y):
    n = len(y)
    if n < 3: return 0.0
    energy = 0.0
    for i in range(1, n-1):
        d2 = (y[i+1] - 2*y[i] + y[i-1])
        energy += d2 * d2
    return np.sqrt(energy / (n-2))

@jit(nopython=True, cache=True)
def _rolling_fractal_energy(arr, window):
    n = len(arr); out = np.full(n, 0.0)
    if window >= n: return out
    for i in range(window - 1, n):
        out[i] = _fractal_energy_nb(arr[i - window + 1 : i + 1])
    return out

# Warmup
_d = np.random.randn(100).astype(np.float64)
_h = np.random.randn(100).astype(np.float64) + 10
_l = _h - np.random.rand(100).astype(np.float64)
_fib = np.array([1.0, 1.0, 2.0, 3.0, 5.0, 8.0, 13.0])
_fast_linslope(_d, 10); _fast_zscore(_d, 10); _fast_shannon(_d, 20)
_fast_hurst(_d, 50); _fast_tema(_d); _fast_r_sq(_d, 30)
_fast_wma(_d, 10); _fast_fwma(_d, _fib); _fast_polynomial_fit(_d, 60); _fast_mass_index(_h, _l, 25)
_kalman_numba(_d); _rolling_cog(_d, 20); _vhf_nb(_d, 28); _psar_nb(_d, _d, _d)
_choppiness_nb(_d, _d, _d, 14); _rolling_fractal_energy(_d, 20)
print("✅ Numba kernels ready (37 indicators compiled)\n")

# ==============================================================================
# MODULAR FEATURE SEEDS (TREND FAMILY - PHYSICS-FIRST)
# Classification: Cumulative Pillars vs Oscillators/Ratios
# ==============================================================================
def build_feature_seeds(h, l, c, o, v, idx):
    """
    TREND FAMILY - PHYSICS-FIRST ARCHITECTURE (COMPLETE WITH RESEARCH)

    CLASSIFICATION PROTOCOL:
    - Cumulative Pillars (trend to infinity) → Rolling % or Log-Diff
    - Oscillators/Ratios (stationary) → Z-score Lenses (z, slope, sos)

    KINEMATIC MANDATE:
    - Level 2 (Position): 90-day Z-score centers on origin
    - Level 3 (Velocity): Slope of Z-score (trend strength)
    - Level 4 (Acceleration): Slope-of-slope (turning points)
    - Level 10 (Fast Turning): 10-day slope (early momentum shifts)

    FEATURE COUNT:
    - 37 z_lens seeds × 6 transforms = 222 Z_LENS features
    - 3 win seeds × 2 transforms = 6 WIN features
    - TOTAL: 228 raw features → PCA to 19 components

    RESEARCH-VALIDATED ADDITIONS (13 new):
    - HMA (Hull Moving Average) - Fast trend detection
    - FWMA (Fibonacci WMA) - Lag-optimized tracking
    - Medium Filter - High-vol smoothing
    - Velocity & Acceleration - Physics of the move
    - EMA 50/200 - Regime shifts
    - Trend Factors (4 horizons) - Multi-timeframe normalization
    - Mass Index - Range contraction reversals
    - MA Distance - Standardized drift measurement

    Returns 2 dicts: z_lens_seeds (37), win_seeds (3)
    """
    hlc = (h + l + c) / 3
    n = len(hlc)
    hlc_s = pd.Series(hlc, index=idx)
    c_s = pd.Series(c, index=idx)

    # ═══════════════════════════════════════════════════════════════════════
    # Z_LENS INDICATORS (Stationary Oscillators/Ratios)
    # Get: z-score, slope, sos on 10-day & 90-day windows
    # ═══════════════════════════════════════════════════════════════════════

    # 1. STATIONARITY: Log differencing
    # Formula: log(price_t) - log(price_t-1)
    log_hlc = np.log(hlc + 1e-9)
    log_diff = np.zeros(n)
    log_diff[1:] = log_hlc[1:] - log_hlc[:-1]

    # 2. INTERACTION RATIOS: HLC3 / Smoothed Anchors
    # "Superior way to depict non-linear behavior while mitigating multicollinearity"

    # TEMA (Triple Exponential MA - minimizes lag)
    # Formula: TEMA = 3*EMA1 - 3*EMA2 + EMA3
    tema_30 = _fast_tema(hlc)  # Uses existing Numba-optimized function
    tema_30_ratio = hlc / (tema_30 + 1e-9)

    # TEMA 10 (short-term)
    ema1_10 = hlc_s.ewm(span=10).mean().values
    ema2_10 = pd.Series(ema1_10, index=idx).ewm(span=10).mean().values
    ema3_10 = pd.Series(ema2_10, index=idx).ewm(span=10).mean().values
    tema_10 = 3*ema1_10 - 3*ema2_10 + ema3_10
    tema_10_ratio = hlc / (tema_10 + 1e-9)

    # SMA ratios
    sma_5 = hlc_s.rolling(5).mean().values
    sma_5_ratio = hlc / (sma_5 + 1e-9)

    sma_20 = hlc_s.rolling(20).mean().values
    sma_20_ratio = hlc / (sma_20 + 1e-9)

    # VIDYA (Variable Index Dynamic Average - CMO-adaptive)
    # Formula: VIDYA[i] = alpha * |CMO[i]| * price[i] + (1 - alpha * |CMO[i]|) * VIDYA[i-1]
    cmo_10 = (hlc_s.diff().rolling(10).sum() / (hlc_s.diff().abs().rolling(10).sum() + 1e-9)).values
    abs_cmo = np.abs(cmo_10)
    alpha = 2.0 / (10 + 1)
    vidya_10 = np.zeros(n)
    vidya_10[0] = hlc[0]
    for i in range(1, n):
        vidya_10[i] = alpha * abs_cmo[i] * hlc[i] + (1 - alpha * abs_cmo[i]) * vidya_10[i-1]
    vidya_10_ratio = hlc / (vidya_10 + 1e-9)

    # Kalman Filter (adaptive smoothing) - used as RATIO, not raw
    # Formula: Recursive Bayesian with fixed noise (r=0.0001, q=0.001)
    kalman = _kalman_numba(hlc)
    kalman_ratio = hlc / (kalman + 1e-9)

    # MTSI (Modified True Strength Index - close vs VWAP)
    # Formula: EMA(EMA(close - VWAP, 3), 2)
    vwap = np.cumsum(hlc * v) / (np.cumsum(v) + 1e-9)
    mtsi_input = c - vwap
    mtsi_ema1 = pd.Series(mtsi_input, index=idx).ewm(span=3).mean().values
    mtsi = pd.Series(mtsi_ema1, index=idx).ewm(span=2).mean().values

    # 3. EFFICIENCY ESTIMATORS (Bounded, scale-invariant)

    # Efficiency Ratio (Kaufman's ER)
    # Formula: ER = |Direction| / Volatility
    # "Measures lack of dynamic equilibrium"
    er_20 = (hlc_s.diff(20).abs() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values

    # VIDYA CMO (momentum component)
    vidya_cmo = (hlc_s.diff().rolling(20).sum() / (hlc_s.diff().abs().rolling(20).sum() + 1e-9)).values

    # 4. FRACTAL EFFICIENCY (Hurst Exponent)
    # Formula: H from rescaled range (R/S) analysis
    # "Predictability positively correlated with H; H>0.5 = long-term memory"
    hurst_50 = _fast_hurst(hlc, 50)
    hurst_20 = _fast_hurst(hlc, 20)

    # 5. ENTROPY (Shannon)
    # Formula: -sum(p_i * log2(p_i)) over probability bins
    # "Decreasing entropy confirms decrease in efficiency during crisis"
    shannon_20 = _fast_shannon(hlc, 20)
    shannon_10 = _fast_shannon(hlc, 10)

    # 6. TREND QUALITY (R-squared)
    # Formula: Correlation coefficient squared from linear regression
    r_sq_30 = _fast_r_sq(hlc, 30)
    r_sq_10 = _fast_r_sq(hlc, 10)
    linreg_slope_30 = _fast_linslope(hlc, 30)

    # 7. TREND STRENGTH (ADX)
    tr = np.maximum(h - l, np.maximum(np.abs(h - np.roll(c, 1)), np.abs(l - np.roll(c, 1))))
    tr[0] = h[0] - l[0]
    atr_14 = pd.Series(tr, index=idx).rolling(14).mean().values
    dm_plus = np.maximum(h - np.roll(h, 1), 0); dm_plus[0] = 0
    dm_minus = np.maximum(np.roll(l, 1) - l, 0); dm_minus[0] = 0
    pdi = 100 * pd.Series(dm_plus, index=idx).rolling(14).mean().values / (atr_14 + 1e-9)
    mdi = 100 * pd.Series(dm_minus, index=idx).rolling(14).mean().values / (atr_14 + 1e-9)
    adx = (100 * np.abs(pdi - mdi) / (pdi + mdi + 1e-9))
    adx = pd.Series(adx, index=idx).rolling(14).mean().values

    # 8. CENTER OF GRAVITY (Ehlers)
    # Classification: STABLE OSCILLATOR (not cumulative pillar)
    # Formula: COG = -sum(w_i * price_i) / sum(price_i), w_i = position weight
    # "Identifies turning points almost instantaneously"
    # Treatment: Z_LENS (90-day for position, slopes for velocity/acceleration)
    cog_20 = _rolling_cog(hlc, 20)

    # 9. COMPOSITE RATIOS (stationary interaction terms)
    shannon_ratio_10_20 = shannon_10 / (shannon_20 + 1e-9)
    entropy_r_sq_ratio = shannon_20 / (r_sq_30 + 1e-9)
    r_sq_ratio_10_30 = r_sq_10 / (r_sq_30 + 1e-9)
    er_10 = (hlc_s.diff(10).abs() / (hlc_s.diff().abs().rolling(10).sum() + 1e-9)).values
    er_ratio_10_20 = er_10 / (er_20 + 1e-9)

    r_sq_hurst_ratio = r_sq_30 / (hurst_50 + 1e-9)
    vhf_adx_ratio = vhf_28 / (adx/100.0 + 1e-9)
    chop_hurst_ratio = choppiness_14 / (hurst_50*100.0 + 1e-9)
    fractal_shannon_ratio = fractal_energy_20 / (shannon_20 + 1e-9)
    vhf_chop_ratio = vhf_28 / (choppiness_14/100.0 + 1e-9)

    sma_30 = hlc_s.rolling(30).mean().values
    dispersion = np.std(np.stack([hlc / (sma_30 + 1e-9), hlc / (tema_30 + 1e-9)], axis=1), axis=1)
    dispersion_r_sq = dispersion / (r_sq_30 + 1e-9)

    # ═══════════════════════════════════════════════════════════════════════
    # RESEARCH-VALIDATED ADDITIONS (13 new indicators)
    # ═══════════════════════════════════════════════════════════════════════

    # 10. HULL MOVING AVERAGE (HMA)
    # Formula: HMA = WMA(2*WMA(n/2) - WMA(n), sqrt(n))
    # "Identifies emerging trends swiftly by reducing inherent delay"
    n_period = 21
    wma_half = _fast_wma(hlc, n_period // 2)
    wma_full = _fast_wma(hlc, n_period)
    hma_raw = 2 * wma_half - wma_full
    hma_21 = _fast_wma(hma_raw, int(np.sqrt(n_period)))
    hma_21_ratio = hlc / (hma_21 + 1e-9)

    # 11. FIBONACCI WEIGHTED MOVING AVERAGE (FWMA)
    # Formula: FWMA = sum(Close_i * F_i) / sum(F_i)
    # "Uniquely combines smoothing with Fibonacci weighting for lag-optimized tracking"
    fib_len = 13
    fib_weights = np.zeros(fib_len)
    fib_weights[0] = 1.0
    fib_weights[1] = 1.0
    for i in range(2, fib_len):
        fib_weights[i] = fib_weights[i-1] + fib_weights[i-2]
    fwma_13 = _fast_fwma(hlc, fib_weights)
    fwma_13_ratio = hlc / (fwma_13 + 1e-9)

    # 12. "MEDIUM" FILTER
    # Formula: Medium = (High + Low) / 2
    # "Low-pass filter that smooths wicks/outliers, enhances generalization"
    medium = (h + l) / 2
    medium_hlc_ratio = hlc / (medium + 1e-9)

    # 13-14. PRICE VELOCITY & ACCELERATION (Physics of the Move)
    # Formula: Fit S(t) = A0 + A1*t + A2*t^2 to price series
    # Velocity ∝ A1, Acceleration ∝ A2
    # "Provides the Physics of the Move required to detect turning points"
    velocity_60, acceleration_60 = _fast_polynomial_fit(hlc, 60)

    # 15. EMA 50 / EMA 200 RATIO
    # Formula: Ratio = EMA50(Close) / EMA200(Close)
    # "Encodes short/long-term relationship for regime shift detection"
    ema_50 = c_s.ewm(span=50).mean().values
    ema_200 = c_s.ewm(span=200).mean().values
    ema_50_200_ratio = ema_50 / (ema_200 + 1e-9)

    # 16-19. TREND FACTOR (Multi-Horizon)
    # Formula: TrendFactor_{t,L} = Price_t / MA_L(Price)_t
    # "Normalizes across horizons from 3 to 1000 days"
    trend_factor_10 = hlc / (hlc_s.rolling(10).mean().values + 1e-9)
    trend_factor_30 = hlc / (hlc_s.rolling(30).mean().values + 1e-9)
    trend_factor_100 = hlc / (hlc_s.rolling(100).mean().values + 1e-9)
    trend_factor_200 = hlc / (hlc_s.rolling(200).mean().values + 1e-9)

    # 20. MASS INDEX
    # Formula: MI = sum(EMA9(High-Low) / EMA9(EMA9(High-Low)))
    # "Identifies reversals when range contracts after expansion"
    mass_index = _fast_mass_index(h, l, 25)

    # 21. MOVING AVERAGE DISTANCE (Standardized)
    # "Measures how far price has drifted from established trend"
    sma_60 = hlc_s.rolling(60).mean().values
    ma_distance_60 = (hlc - sma_60) / (sma_60 + 1e-9)
    ma_dist_mean = pd.Series(ma_distance_60, index=idx).rolling(60).mean().values
    ma_dist_std = pd.Series(ma_distance_60, index=idx).rolling(60).std().values
    ma_distance_standardized = (ma_distance_60 - ma_dist_mean) / (ma_dist_std + 1e-9)

    # ═══════════════════════════════════════════════════════════════════════
    # WIN INDICATORS (Unbounded metrics - need rolling % normalization)
    # ═══════════════════════════════════════════════════════════════════════

    # VHF (Vertical Horizontal Filter - trend strength ratio)
    # Formula: VHF = (max(high) - min(low)) / sum(|close[i] - close[i-1]|)
    vhf_28 = _vhf_nb(hlc, 28)
    choppiness_14 = _choppiness_nb(h, l, c, 14)
    fractal_energy_20 = _rolling_fractal_energy(hlc, 20)

    # PSAR (Parabolic SAR)
    # Formula: PSAR[i] = PSAR[i-1] + AF * (EP - PSAR[i-1])
    psar_values, psar_trend = _psar_nb(h, l, c)
    psar_distance = (c - psar_values) / (psar_values + 1e-9)

    # Return seeds organized by treatment type
    z_lens_seeds = {
        # Stationarity
        'log_diff': log_diff,

        # Interaction ratios (HLC3 / smoothed anchors)
        'tema_30_ratio': tema_30_ratio,
        'tema_10_ratio': tema_10_ratio,
        'sma_5_ratio': sma_5_ratio,
        'sma_20_ratio': sma_20_ratio,
        'vidya_10_ratio': vidya_10_ratio,
        'kalman_ratio': kalman_ratio,

        # VWAP-based
        'mtsi': mtsi,

        # Efficiency estimators
        'er_20': er_20,
        'vidya_cmo': vidya_cmo,

        # Fractal efficiency
        'hurst_50': hurst_50,
        'hurst_20': hurst_20,

        # Entropy
        'shannon_20': shannon_20,
        'shannon_10': shannon_10,

        # Trend quality
        'r_sq_30': r_sq_30,
        'r_sq_10': r_sq_10,
        'linreg_slope_30': linreg_slope_30,

        # Trend strength
        'adx_14': adx,

        # Center of Gravity (MOVED from WIN to Z_LENS per Physics-First mandate)
        'cog_20': cog_20,

        # Composite ratios
        'shannon_ratio_10_20': shannon_ratio_10_20,
        'entropy_r_sq_ratio': entropy_r_sq_ratio,
        'r_sq_ratio_10_30': r_sq_ratio_10_30,
        'er_ratio_10_20': er_ratio_10_20,
        'dispersion_r_sq': dispersion_r_sq,
        'r_sq_hurst_ratio': r_sq_hurst_ratio,
        'vhf_adx_ratio': vhf_adx_ratio,
        'chop_hurst_ratio': chop_hurst_ratio,
        'fractal_shannon_ratio': fractal_shannon_ratio,
        'vhf_chop_ratio': vhf_chop_ratio,

        # ═══ RESEARCH-VALIDATED ADDITIONS ═══
        # Advanced trend indicators from academic research

        # Hull MA (fast trend detection)
        'hma_21_ratio': hma_21_ratio,

        # Fibonacci WMA (lag-optimized)
        'fwma_13_ratio': fwma_13_ratio,

        # Medium filter (high-vol smoothing)
        'medium_filter_ratio': medium_hlc_ratio,

        # Physics: Velocity & Acceleration
        'price_velocity_60': velocity_60,
        'price_acceleration_60': acceleration_60,

        # EMA 50/200 (regime shifts)
        'ema_50_200_ratio': ema_50_200_ratio,

        # Multi-horizon trend factors
        'trend_factor_10': trend_factor_10,
        'trend_factor_30': trend_factor_30,
        'trend_factor_100': trend_factor_100,
        'trend_factor_200': trend_factor_200,

        # Mass Index (range reversals)
        'mass_index_25': mass_index,

        # MA Distance (standardized drift)
        'ma_distance_60_std': ma_distance_standardized,
    }

    win_seeds = {
        'vhf_28': vhf_28,
        'psar_distance': psar_distance,
        'psar_trend': psar_trend,
    }

    return z_lens_seeds, win_seeds

# ==============================================================================
# FEATURE FACTORY
# ==============================================================================
def generate_features(df, brain_name='DIRECTION', ease_data=None):
    """
    Generate features with correct targets for each brain
    Applies different lens treatments based on indicator type
    """
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.copy()
    df.columns = [c.lower() for c in df.columns]

    h = df['high'].values.astype(np.float64)
    l = df['low'].values.astype(np.float64)
    c = df['close'].values.astype(np.float64)
    o = df['open'].values.astype(np.float64) if 'open' in df.columns else c.copy()
    v = df['volume'].values.astype(np.float64)
    idx = df.index

    # Get seeds from modular builder (returns 2 dicts by lens type)
    z_lens_seeds, win_seeds = build_feature_seeds(h, l, c, o, v, idx)

    # ═══════════════════════════════════════════════════════════════════════
    # APPLY Z_LENS TRANSFORMS (Physics-First: Position, Velocity, Acceleration)
    # ═══════════════════════════════════════════════════════════════════════
    # Level 2 (Position): Z-score on 10-day & 90-day windows
    # Level 3 (Velocity): Slope of Z-score (trend strength)
    # Level 4 (Acceleration): Slope-of-slope (turning points)
    # Level 10 (Fast Turning): 10-day captures early momentum shifts

    for name, arr in z_lens_seeds.items():
        for lens in [10, 90]:
            rm = pd.Series(arr, index=idx).rolling(lens).mean().values
            rs = pd.Series(arr, index=idx).rolling(lens).std().values
            z = (arr - rm) / (rs + 1e-9)
            z_slope = _fast_linslope(z, lens)
            z_sos = _fast_linslope(z_slope, lens)
            df[f'LENS_{lens}_{name}_z'] = z
            df[f'LENS_{lens}_{name}_z_slope'] = z_slope
            df[f'LENS_{lens}_{name}_z_sos'] = z_sos

    # ═══════════════════════════════════════════════════════════════════════
    # APPLY WIN TRANSFORMS (rolling % for unbounded metrics)
    # ═══════════════════════════════════════════════════════════════════════
    for name, arr in win_seeds.items():
        for win in [10, 30]:
            rm = pd.Series(arr, index=idx).rolling(win).mean().values
            df[f'WIN_{win}_{name}_pct'] = arr / (rm + 1e-9) - 1

    # TARGETS (Brain-specific)
    if brain_name == 'DIRECTION':
        # Binary: Tomorrow close > today close
        df['T_FINAL'] = (pd.Series(c, index=idx).shift(-1) > c).astype(int)
        df['T_FINAL'].iloc[-1] = np.nan

    elif brain_name == 'EASE':
        # Regression: Tomorrow's ease value
        if ease_data is not None:
            # Merge with EASE parquet
            ease_data = ease_data.copy()
            ease_data.index = pd.to_datetime(ease_data.index)
            df_with_ease = df.join(ease_data[['EASE_val']], how='left')
            df['T_FINAL'] = df_with_ease['EASE_val'].shift(-1)
        else:
            # Fallback: normalized price movement
            daily_range = h - l
            avg_range = pd.Series(daily_range, index=idx).rolling(14).mean().values
            price_move = (c - o) / (avg_range + 1e-9)
            df['T_FINAL'] = pd.Series(price_move, index=idx).shift(-1)
        df['T_FINAL'].iloc[-1] = np.nan

    elif brain_name == 'EXP':
        # Regression: Tomorrow's range expansion
        daily_range = h - l
        avg_range_20 = pd.Series(daily_range, index=idx).rolling(20).mean()
        range_tomorrow = pd.Series(daily_range, index=idx).shift(-1)
        df['T_FINAL'] = range_tomorrow / (avg_range_20 + 1e-9)
        df['T_FINAL'].iloc[-1] = np.nan

    # Cleanup
    df = df.replace([np.inf, -np.inf], np.nan)

    # First, must have valid target
    df = df.dropna(subset=['T_FINAL'])

    # For features: only drop rows where >50% are NaN (too many missing)
    # This allows indicators with long warmup (EMA 200, etc.) to not kill all data
    feature_cols = [c for c in df.columns if c.startswith('LENS_') or c.startswith('WIN_')]
    nan_threshold = len(feature_cols) * 0.5  # Drop if >50% features are NaN
    df = df.dropna(subset=feature_cols, thresh=len(feature_cols) - int(nan_threshold))

    # Forward-fill remaining NaNs in features (conservative - uses past values only)
    df[feature_cols] = df[feature_cols].ffill()

    # If any NaNs remain at the start (before first valid value), fill with 0
    df[feature_cols] = df[feature_cols].fillna(0)

    if len(df) < 100:
        return pd.DataFrame()  # Not enough data after cleaning

    return df

# ==============================================================================
# DATA LOADING (SIMPLE SERIAL FOR CPU)
# ==============================================================================
def load_data(symbols, period, brain_name, ease_data=None):
    """Simple serial loading - works on CPU"""
    print(f"📥 Loading {len(symbols)} symbols ({period})...")

    all_dfs = []

    for sym in tqdm(symbols, desc="Processing"):
        try:
            df = yf.download(sym, period=period, progress=False, auto_adjust=True, threads=False)
            min_rows = 400 if not FAST_MODE else 250  # Lower requirement for fast mode (shorter sequences)
            if df.empty or len(df) < min_rows:
                continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.columns = [c.lower() for c in df.columns]

            if not all(c in df.columns for c in ['high', 'low', 'close']):
                continue

            result = generate_features(df, brain_name, ease_data)
            if not result.empty and len(result) >= 100:  # Need at least 100 rows after cleaning
                result['symbol'] = sym
                all_dfs.append(result)

        except Exception as e:
            continue

    if not all_dfs:
        return pd.DataFrame()

    master = pd.concat(all_dfs, axis=0, ignore_index=True)
    print(f"   ✅ Master: {len(master):,} rows from {len(all_dfs)} symbols\n")
    return master

# ==============================================================================
# MODEL BUILDING
# ==============================================================================
def build_model(n_features, brain_name, seq_len):
    """Build model based on brain type"""
    model_type = MODEL_TYPE[brain_name]
    activation = ACTIVATION[brain_name]
    loss_type = LOSS_TYPE[brain_name]

    # Smaller model in fast mode for speed
    if FAST_MODE:
        units_1, units_2, dense_units = 64, 32, 16
    else:
        units_1, units_2, dense_units = 128, 64, 32

    with tf.device(DEVICE):
        model = Sequential([
            Input(shape=(seq_len, n_features)),
            GRU(units_1, return_sequences=True) if model_type == 'GRU' else LSTM(units_1, return_sequences=True),
            Dropout(0.2),
            GRU(units_2) if model_type == 'GRU' else LSTM(units_2),
            Dropout(0.2),
            Dense(dense_units, activation='relu'),
            Dense(1, activation=activation, dtype='float32')
        ])

        loss = tf.keras.losses.Huber(delta=1.0) if loss_type == 'huber' else loss_type
        metrics = ['accuracy'] if brain_name == 'DIRECTION' else ['mae']

        model.compile(optimizer=Adam(1e-3), loss=loss, metrics=metrics)

    return model

# ==============================================================================
# TRAINING & EVALUATION
# ==============================================================================
def run_audit(master_df, brain_name):
    """Train and evaluate model"""
    seq_len = SEQ_LEN
    epochs = EPOCHS
    batch_size = BATCH_SIZE

    feature_cols = [c for c in master_df.columns if c.startswith('LENS_') or c.startswith('WIN_')]

    z_lens_count = len([c for c in feature_cols if c.startswith('LENS_')])
    win_count = len([c for c in feature_cols if c.startswith('WIN_')])

    print(f"  [FEATURES] {len(feature_cols)} total ({z_lens_count} LENS + {win_count} WIN → PCA to 19)")

    X_raw = master_df[feature_cols].values.astype(np.float32)
    y_raw = master_df['T_FINAL'].values.astype(np.float32)

    # Create sequences
    n = len(X_raw)
    X_seqs = np.stack([X_raw[i-seq_len:i] for i in range(seq_len, n)])
    y_seqs = y_raw[seq_len:]

    # Walk-forward splits
    train_end = int(len(X_seqs) * 0.70)
    val_end = int(len(X_seqs) * 0.85)

    X_tr = X_seqs[:train_end]
    y_tr = y_seqs[:train_end]
    X_val = X_seqs[train_end:val_end]
    y_val = y_seqs[train_end:val_end]
    X_test = X_seqs[val_end:]
    y_test = y_seqs[val_end:]

    print(f"  [SPLITS] Train={len(X_tr):,} | Val={len(X_val):,} | Test={len(X_test):,}")

    # Scale (fit on train only - anti-leakage)
    scaler = RobustScaler()
    n_train, seq, feats = X_tr.shape
    X_tr_2d = X_tr.reshape(-1, feats)

    # Warmup exclusion (first 350 rows)
    warmup = min(350, len(X_tr_2d) // 2)
    scaler.fit(X_tr_2d[warmup:])

    X_tr_scaled = scaler.transform(X_tr_2d).reshape(n_train, seq, feats)
    X_val_scaled = scaler.transform(X_val.reshape(-1, feats)).reshape(len(X_val), seq, feats)
    X_test_scaled = scaler.transform(X_test.reshape(-1, feats)).reshape(len(X_test), seq, feats)

    # PCA (57 → 19 components)
    print(f"  [PCA] {feats} → 19 components...")
    pca = PCA(n_components=19)
    X_tr_flat = X_tr_scaled.reshape(-1, feats)
    pca.fit(X_tr_flat[warmup:])

    var_explained = np.sum(pca.explained_variance_ratio_)
    print(f"  [PCA] Variance retained: {var_explained*100:.1f}%")

    X_tr_pca = pca.transform(X_tr_scaled.reshape(-1, feats)).reshape(n_train, seq, 19)
    X_val_pca = pca.transform(X_val_scaled.reshape(-1, feats)).reshape(len(X_val), seq, 19)
    X_test_pca = pca.transform(X_test_scaled.reshape(-1, feats)).reshape(len(X_test), seq, 19)

    # ═══════════════════════════════════════════════════════════════════════
    # FEATURE IMPORTANCE ANALYSIS
    # ═══════════════════════════════════════════════════════════════════════

    # 1. PCA Component Loadings (which raw features → which components)
    loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f'PC{i+1}' for i in range(19)],
        index=feature_cols
    )

    # 2. Feature contribution to PCA (sum of absolute loadings across all components)
    feature_importance = loadings.abs().sum(axis=1).sort_values(ascending=False)

    # 3. Aggregate to SEEDS (combine all lens transforms per indicator)
    # Extract seed name from feature column (e.g., 'LENS_10_hma_21_ratio_z' → 'hma_21_ratio')
    seed_importance = {}
    for feat, imp in feature_importance.items():
        if feat.startswith('LENS_'):
            # Format: LENS_{window}_{seed_name}_{transform}
            parts = feat.split('_')
            seed_name = '_'.join(parts[2:-1])  # Everything between window and transform
        elif feat.startswith('WIN_'):
            # Format: WIN_{window}_{seed_name}_pct
            parts = feat.split('_')
            seed_name = '_'.join(parts[2:-1])  # Everything between window and _pct
        else:
            seed_name = feat

        if seed_name not in seed_importance:
            seed_importance[seed_name] = 0
        seed_importance[seed_name] += imp

    seed_importance_df = pd.DataFrame([
        {'seed': k, 'total_importance': v, 'avg_importance': v / (6 if k in ['vhf_28', 'psar_distance', 'psar_trend'] else 6)}
        for k, v in sorted(seed_importance.items(), key=lambda x: x[1], reverse=True)
    ])

    print(f"\n  [TOP 10 SEEDS BY IMPORTANCE]")
    for idx, row in seed_importance_df.head(10).iterrows():
        print(f"    {idx+1:2d}. {row['seed']:<25} {row['total_importance']:>8.2f}")

    print(f"\n  [BOTTOM 10 SEEDS BY IMPORTANCE]")
    for idx, row in seed_importance_df.tail(10).iterrows():
        print(f"    {idx+1:2d}. {row['seed']:<25} {row['total_importance']:>8.2f}")

    # ═══════════════════════════════════════════════════════════════════════
    # IMPORTANCE-ONLY MODE: Skip training, just return feature rankings
    # ═══════════════════════════════════════════════════════════════════════
    if IMPORTANCE_ONLY:
        print(f"\n  ⚡ IMPORTANCE-ONLY MODE: Skipping training (rankings calculated)")
        # Return dummy metrics since we didn't train
        return 0.0, 0.0, seed_importance_df

    # Build model
    model = build_model(19, brain_name, seq_len)

    print(f"\n  [TRAIN] {epochs} epochs...")

    # Train
    with tf.device(DEVICE):
        model.fit(X_tr_pca, y_tr,
                  validation_data=(X_val_pca, y_val),
                  epochs=epochs,
                  batch_size=batch_size,
                  callbacks=[
                      EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE,
                                  restore_best_weights=True),
                      ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                      patience=LR_REDUCE_PATIENCE, min_lr=1e-6)
                  ],
                  verbose=1)

    # Evaluate
    is_classification = (brain_name == 'DIRECTION')

    if is_classification:
        _, val_metric = model.evaluate(X_val_pca, y_val, verbose=0)
        _, test_metric = model.evaluate(X_test_pca, y_test, verbose=0)
        metric_name = 'Accuracy'
    else:
        _, val_metric = model.evaluate(X_val_pca, y_val, verbose=0)
        _, test_metric = model.evaluate(X_test_pca, y_test, verbose=0)
        metric_name = 'MAE'

    print(f"\n  [RESULTS]")
    print(f"    Val {metric_name}:  {val_metric:.4f}")
    print(f"    Test {metric_name}: {test_metric:.4f}")

    # Quality gates
    if is_classification:
        if val_metric > 0.70 and test_metric > 0.70:
            print(f"    🚨 SEVERE LEAKAGE (both >70%)")
        elif val_metric > 0.70:
            print(f"    ⚠️  High val (overfit, acceptable)")
        elif test_metric > 0.50:
            print(f"    ✅ Signal detected")
        else:
            print(f"    ⚠️  Weak signal")

    del model
    gc.collect()
    tf.keras.backend.clear_session()

    return val_metric, test_metric, seed_importance_df

# ==============================================================================
# MAIN
# ==============================================================================
if __name__ == '__main__':
    print("\n" + "="*70)
    print("   SOVEREIGN TITAN v4.0 — SCOTTIE PIPPEN EDITION")
    if IMPORTANCE_ONLY:
        print("   🔍 IMPORTANCE-ONLY: Just ranking features (~30 sec per iteration)")
    elif FAST_MODE:
        print("   ⚡ FAST MODE: 20 epochs | 2048 batch | 30 seq | 64/32 units")
    else:
        print("   🐢 FULL MODE: 50 epochs | 512 batch | 60 seq | 128/64 units")
    print("="*70)
    print("  🧠 DIRECTION: Tomorrow close > today (binary, GRU)")
    print("  🧠 EASE: Tomorrow's 3min tradability (regression, LSTM)")
    print("  🧠 EXP: Tomorrow's range expansion (regression, LSTM)")
    print("="*70 + "\n")

    choice = input("Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
    BRAINS_TO_RUN = ['DIRECTION','EASE','EXP'] if choice == '4' else [
        {'1':'DIRECTION','2':'EASE','3':'EXP'}[choice]
    ]

    num_symbols = int(input("Symbols [20]: ") or "20")
    num_iters = int(input("Iterations [5]: ") or "5")

    # Load EASE data if needed
    ease_data = None
    if 'EASE' in BRAINS_TO_RUN:
        try:
            ease_data = pd.read_parquet(EASE_PARQUET_PATH)
            print(f"✅ EASE data: {len(ease_data)} rows\n")
        except Exception:
            print(f"⚠️  No EASE data at {EASE_PARQUET_PATH} (using fallback)\n")

    all_results = []
    all_feature_importance = []

    for BRAIN in BRAINS_TO_RUN:
        print(f"\n{'='*70}")
        print(f"  BRAIN: {BRAIN} | {MODEL_TYPE[BRAIN]} | {DATA_WINDOW[BRAIN]}")
        print(f"{'='*70}")

        for it in range(1, num_iters + 1):
            print(f"\n{'─'*70}")
            print(f"  ITERATION {it}/{num_iters}")
            print(f"{'─'*70}\n")

            pool = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
            master_df = load_data(pool, DATA_WINDOW[BRAIN], BRAIN, ease_data)

            if master_df.empty:
                print("  ⚠️  No data - skipping\n")
                continue

            val_metric, test_metric, seed_importance_df = run_audit(master_df, BRAIN)

            # Quality gate for DIRECTION (skip in importance-only mode)
            if not IMPORTANCE_ONLY and BRAIN == 'DIRECTION' and val_metric < 0.50:
                print(f"  ⚠️  Val {val_metric:.4f} < 0.50 - skipping\n")
                gc.collect()
                continue

            # Store metrics only if we actually trained
            if not IMPORTANCE_ONLY:
                all_results.append({
                    'iteration': it,
                    'brain': BRAIN,
                    'val_metric': val_metric,
                    'test_metric': test_metric,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })

            # Store feature importance with iteration/brain info
            seed_importance_df['iteration'] = it
            seed_importance_df['brain'] = BRAIN
            all_feature_importance.append(seed_importance_df)

            gc.collect()

    # Export feature importance (always available)
    if all_feature_importance:
        feat_df = pd.concat(all_feature_importance, ignore_index=True)
        feat_csv_path = os.path.join(OUTPUT_DIR, f"Feature_Importance_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        feat_df.to_csv(feat_csv_path, index=False)

        print("\n" + "="*70)
        print("✅ COMPLETE")
        print(f"📊 Feature Importance: {feat_csv_path}")

        # Show overall top/bottom seeds across all iterations
        avg_importance = feat_df.groupby('seed')['total_importance'].mean().sort_values(ascending=False)
        print(f"\n{'='*70}")
        print("📊 OVERALL FEATURE IMPORTANCE (Averaged Across Iterations)")
        print(f"{'='*70}")
        print(f"\n🏆 TOP 10 SEEDS:")
        for idx, (seed, imp) in enumerate(avg_importance.head(10).items(), 1):
            print(f"  {idx:2d}. {seed:<30} {imp:>8.2f}")
        print(f"\n⚠️  BOTTOM 10 SEEDS:")
        for idx, (seed, imp) in enumerate(avg_importance.tail(10).items(), 1):
            print(f"  {idx:2d}. {seed:<30} {imp:>8.2f}")
        print(f"{'='*70}\n")

    # Export training results (only if we actually trained)
    if all_results:
        df = pd.DataFrame(all_results)
        csv_path = os.path.join(OUTPUT_DIR, f"Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        df.to_csv(csv_path, index=False)

        print(f"📂 Training Results: {csv_path}\n")

        print("="*70)
        print("📊 RESULTS BY BRAIN:")
        print("="*70)
        for brain in df['brain'].unique():
            brain_data = df[df['brain'] == brain]
            avg_metric = brain_data['test_metric'].mean()

            if brain == 'DIRECTION':
                if avg_metric > 0.70: status = "🚨 LEAKAGE"
                elif avg_metric > 0.50: status = "✅ SIGNAL"
                else: status = "⚠️  NOISE"
                print(f"  {brain:<12} Test Acc: {avg_metric:.4f}  {status}")
            else:
                print(f"  {brain:<12} Test MAE: {avg_metric:.4f}")
        print("="*70)
    elif IMPORTANCE_ONLY:
        print("⚡ IMPORTANCE-ONLY MODE: Training skipped for speed")
        print("   Set IMPORTANCE_ONLY = False to get accuracy metrics")
    else:
        print("\n⚠️ No training data collected")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[ENV] COLAB
[OUTPUT] /content/drive/MyDrive/Sovereign_Titan_Results/Sovereign_Titan_v4_Complete_Research

⚠️  No GPU - using CPU (slower but works)
[DEVICE] /cpu:0

⚡ FAST MODE ENABLED - Training will be 3-5x faster
✅ Numba kernels ready (37 indicators compiled)


   SOVEREIGN TITAN v4.0 — COMPLETE WITH RESEARCH
   ⚡ FAST MODE: 20 epochs | 2048 batch | 30 seq | 64/32 units
  🧠 DIRECTION: Tomorrow close > today (binary, GRU)
  🧠 EASE: Tomorrow's 3min tradability (regression, LSTM)
  🧠 EXP: Tomorrow's range expansion (regression, LSTM)

Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 1
Symbols [20]: 50
Iterations [5]: 5

  BRAIN: DIRECTION | GRU | 4y

──────────────────────────────────────────────────────────────────────
  ITERATION 1/5
──────────────────────────────────────────────────────────────────────

📥 Loading 50 symbols (4y)...


Processing:   0%|          | 0/50 [00:00<?, ?it/s]

   ✅ Master: 49,699 rows from 50 symbols

  [FEATURES] 222 total (216 LENS + 6 WIN → PCA to 19)
  [SPLITS] Train=34,768 | Val=7,450 | Test=7,451
  [PCA] 222 → 19 components...
  [PCA] Variance retained: 100.0%

  [TOP 10 SEEDS BY IMPORTANCE]
     1. hma_21_ratio_z                8.56
     2. fwma_13_ratio_z               7.57
     3. hma_21_ratio                  4.55
     4. entropy_r_sq_ratio_z          4.41
     5. shannon_ratio_10_20_z         2.87
     6. entropy_r_sq_ratio            2.65
     7. r_sq_ratio_10_30              2.34
     8. fwma_13_ratio                 2.02
     9. r_sq_ratio_10_30_z            0.97
    10. shannon_ratio_10_20           0.05

  [BOTTOM 10 SEEDS BY IMPORTANCE]
    66. linreg_slope_30               0.00
    67. sma_20_ratio                  0.00
    68. tema_30_ratio                 0.00
    69. mass_index_25                 0.00
    70. cog_20                        0.00
    71. mtsi                          0.00
    72. medium_filter_ratio        

Processing:   0%|          | 0/50 [00:00<?, ?it/s]

   ✅ Master: 49,699 rows from 50 symbols

  [FEATURES] 222 total (216 LENS + 6 WIN → PCA to 19)
  [SPLITS] Train=34,768 | Val=7,450 | Test=7,451
  [PCA] 222 → 19 components...
  [PCA] Variance retained: 100.0%

  [TOP 10 SEEDS BY IMPORTANCE]
     1. shannon_ratio_10_20_z         6.19
     2. r_sq_ratio_10_30_z            4.79
     3. entropy_r_sq_ratio_z          4.77
     4. hma_21_ratio_z                2.70
     5. entropy_r_sq_ratio            2.66
     6. r_sq_ratio_10_30              2.43
     7. shannon_ratio_10_20           1.75
     8. fwma_13_ratio_z               1.18
     9. hma_21_ratio                  1.02
    10. psar_trend                    0.10

  [BOTTOM 10 SEEDS BY IMPORTANCE]
    66. hurst_50                      0.00
    67. er_20                         0.00
    68. adx_14                        0.00
    69. hurst_20                      0.00
    70. ema_50_200_ratio              0.00
    71. mass_index_25                 0.00
    72. r_sq_10                    

Processing:   0%|          | 0/50 [00:00<?, ?it/s]

   ✅ Master: 49,188 rows from 50 symbols

  [FEATURES] 222 total (216 LENS + 6 WIN → PCA to 19)
  [SPLITS] Train=34,410 | Val=7,374 | Test=7,374
  [PCA] 222 → 19 components...
  [PCA] Variance retained: 100.0%

  [TOP 10 SEEDS BY IMPORTANCE]
     1. hma_21_ratio_z                6.85
     2. shannon_ratio_10_20_z         6.29
     3. hma_21_ratio                  3.30
     4. entropy_r_sq_ratio_z          3.28
     5. fwma_13_ratio_z               3.02
     6. shannon_ratio_10_20           2.90
     7. entropy_r_sq_ratio            2.56
     8. r_sq_ratio_10_30              2.40
     9. r_sq_ratio_10_30_z            0.61
    10. psar_trend                    0.00

  [BOTTOM 10 SEEDS BY IMPORTANCE]
    66. kalman_ratio                  0.00
    67. log_diff                      0.00
    68. price_acceleration_60         0.00
    69. tema_10_ratio                 0.00
    70. mtsi                          0.00
    71. ema_50_200_ratio              0.00
    72. mass_index_25              

KeyboardInterrupt: 